# T-RegGNN v2 (improved) — Time-Aware RegGNN for Dynamic Functional Connectivity

## Self-Contained Notebook for Brain Score Prediction

**Data:** PKL files where `train_graphs.shape = (N_subjects, T_windows)`.  
Each cell contains a PyG `Data` object — one connectome window for one subject.  

**Model:** **T-RegGNN** — per-window DenseGCNConv encoder + LSTM temporal aggregator.  
Each subject = 1 prediction (from the full sequence of T window connectomes).  

**Evaluation:** subject-aggregate Pearson r across 5 inner folds — one prediction per test subject.

## Metric Guidance — Which Metric Should You Use?

This notebook tracks **six regression metrics**.

| Metric | Range | Better | What it measures |
|--------|-------|--------|------------------|
| **MAE** | >= 0 | Lower | Average absolute error in score units |
| **RMSE** | >= 0 | Lower | Same but penalises large errors more |
| **Pearson r** | -1 to 1 | Higher | Linear correlation — standard in neuroimaging |
| **Spearman rho** | -1 to 1 | Higher | Rank correlation — robust to outliers |
| **R** | -1 to 1 | Higher | sqrt(R2) with sign — overall fit magnitude |
| **R2** | -inf to 1 | Higher | % variance explained; R2 < 0 means worse than mean |

### Recommended choices for brain-score prediction

**Use `pearson`** — matches the most common reporting standard in neuroimaging/HCP studies.
It captures whether the predicted rank-order of subjects is correct, independently of scale.

**Use `r2`** — directly answers 'how much variance does my model explain?'  
R2 > 0 means the model beats a trivial mean-prediction baseline. Scale-free.

**Use `mae`** — when you need error in the same units as the score (e.g. IQ points)
and clinical thresholds matter more than explained variance.

**Pitfall:** MAE alone can mislead — a model that always predicts the mean has finite MAE but R2 = 0. Always report at least one error metric AND one correlation metric together.

### How to set your primary metric

In **Step 5 (Config)** set:
```python
PRIMARY_METRIC = 'pearson'   # choices: mae | rmse | pearson | spearman | r | r2
```
This controls:
- Which k is selected as best_k in Step 7
- Which validation metric drives early stopping in Step 8
- Which column is highlighted in the per-k results table


## Step 1: Install Dependencies and Setup Environment


In [ ]:
# ─── Install Dependencies ─────────────────────────────────────────────────────
!pip install torch torch-geometric pymanopt networkx scikit-learn tqdm gdown scipy

# ─── All Imports ──────────────────────────────────────────────────────────────
import os
import random
import pickle
from collections import defaultdict

import numpy as np
import gdown
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import torch_geometric.transforms
from torch_geometric.nn.dense import DenseGCNConv
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline

# ─── CUDA / Device Check ──────────────────────────────────────────────────────
train_on_gpu = torch.cuda.is_available()
print('CUDA is available!  Training on GPU ...' if train_on_gpu else 'CUDA is not available.  Training on CPU ...')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CUDA is available!  Training on GPU ...


## Step 2: Set Outer Fold and Discover All 5 Inner PKL Files

Set `OUTER_FOLD` to 1–5 and `FOLDS_DATA_DIR` to your folder.  
The cell loads **all 5 inner PKL files** for that outer fold and verifies they exist.  
Each PKL already contains its own `train_graphs`, `val_graphs`, `test_graphs` — these are used directly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Set these two values ──────────────────────────────────────────────────────
FOLDS_DATA_DIR = "/content/drive/MyDrive/GNN-mri/folds_data"
OUTER_FOLD     = 1          # legacy default; v15 runner iterates 1..5 itself
N_INNER        = 5          # number of inner folds

# ── Discover and verify all 5 inner PKL files ─────────────────────────────────
inner_pkls = {}
missing    = []
for inner in range(1, N_INNER + 1):
    fname = f"graphs_outer{OUTER_FOLD}_inner{inner}.pkl"
    path  = os.path.join(FOLDS_DATA_DIR, fname)
    inner_pkls[inner] = path
    if not os.path.exists(path):
        missing.append(fname)

if missing:
    raise FileNotFoundError(f"Missing PKL files: {missing}")

print(f"Outer fold  : {OUTER_FOLD}")
print(f"Inner folds : {N_INNER}")
print(f"Found PKL files:")
for inner, path in inner_pkls.items():
    print(f"  inner {inner}: {os.path.basename(path)}")
print("All files verified ✓")


Mounted at /content/drive
Outer fold  : 1
Inner folds : 5
Found PKL files:
  inner 1: graphs_outer1_inner1.pkl
  inner 2: graphs_outer1_inner2.pkl
  inner 3: graphs_outer1_inner3.pkl
  inner 4: graphs_outer1_inner4.pkl
  inner 5: graphs_outer1_inner5.pkl
All files verified ✓


## Step 3: Inspect First Inner Fold (Sanity Check)

Loads `inner1` only to verify the PKL structure before the full run.

In [ ]:
# ─── Inspect inner1 to verify the (N_subjects, T_windows) PKL structure ──────
import numpy as np
_s = torch.load(inner_pkls[1], map_location='cpu', weights_only=False)
_tg = _s['train_graphs']
_vg = _s['val_graphs']
_eg = _s['test_graphs']

# train_graphs is a (N_subj, T_win) numpy array of PyG Data objects
print(f"Fold keys      : {list(_s.keys())}")
print(f"train_graphs   : type={type(_tg).__name__}  shape={_tg.shape}  dtype={_tg.dtype}")
print(f"val_graphs     : shape={_vg.shape}")
print(f"test_graphs    : shape={_eg.shape}")
n_tr_subj, n_win = _tg.shape
print(f"  Subjects in train: {n_tr_subj}  |  Windows per subject: {n_win}")

# Inspect first window of first subject
g0 = _tg[0, 0]
print(f"\nFirst window (subject 0, window 0):")
print(f"  type:        {type(g0).__name__}")
if hasattr(g0, 'x'):          print(f"  x shape:     {tuple(g0.x.shape)}")
if hasattr(g0, 'edge_index'): print(f"  edge_index:  {tuple(g0.edge_index.shape)}")
if hasattr(g0, 'edge_attr') and g0.edge_attr is not None:
                              print(f"  edge_attr:   {tuple(g0.edge_attr.shape)}")
if hasattr(g0, 'y'):          print(f"  y (score):   {g0.y}")
if hasattr(g0, 'num_nodes'):  print(f"  num_nodes:   {g0.num_nodes}")
if hasattr(g0, 'pad'):        print(f"  pad flag:    {g0.pad}")
if hasattr(g0, 'last'):       print(f"  last flag:   {g0.last}")

# Check that y is constant across windows of one subject
ys_subject0 = [_tg[0, t].y for t in range(min(5, n_win))]
print(f"\nFirst 5 windows of subject 0 have y = {ys_subject0}")
print(f"  → y constant across windows? {len(set(map(float, ys_subject0))) == 1}")

# Count padded windows for a few subjects
def _count_real(subj_array):
    real = 0
    for t in range(subj_array.shape[0]):
        g = subj_array[t]
        if not getattr(g, 'pad', False):
            real += 1
    return real

print(f"\nReal (non-padded) windows per subject (first 3 train subjects):")
for i in range(min(3, n_tr_subj)):
    print(f"  subject {i}: {_count_real(_tg[i])}/{n_win}")

del _s, _tg, _vg, _eg
print("\nSanity check passed ✓  (data is (subjects, windows) of connectome graphs)")


Fold keys      : ['train_graphs', 'test_graphs', 'val_graphs', 'train_indices', 'test_indices', 'val_indices']
train_graphs   : type=ndarray  shape=(116, 90)  dtype=object
val_graphs     : shape=(30, 90)
test_graphs    : shape=(37, 90)
  Subjects in train: 116  |  Windows per subject: 90

First window (subject 0, window 0):
  type:        Data
  x shape:     (268, 268)
  edge_index:  (2, 28890)
  edge_attr:   (28890, 1)
  y (score):   4.0
  num_nodes:   268
  pad flag:    False
  last flag:   False

First 5 windows of subject 0 have y = [tensor(4.), tensor(4.), tensor(4.), tensor(4.), tensor(4.)]
  → y constant across windows? True

Real (non-padded) windows per subject (first 3 train subjects):
  subject 0: 90/90
  subject 1: 90/90
  subject 2: 90/90

Sanity check passed ✓  (data is (subjects, windows) of connectome graphs)


## Step 4: Define Conversion and Feature Functions

These functions are called **once per inner fold** inside the main loop (Step 7).  
Each inner fold has its own set of subjects so conversion and pairwise features are rebuilt per fold.

In [ ]:
# ─── Subject-level conversion: temporal mean for sample selection ───────────
def _subject_mean_adjacency(subj_window_array, n_roi):
    """Average dense adjacency across all (non-padded) windows for a subject."""
    acc = torch.zeros((n_roi, n_roi))
    count = 0
    for t in range(subj_window_array.shape[0]):
        g = subj_window_array[t]
        if getattr(g, 'pad', False):
            continue
        # Prefer pre-computed adj if present
        if hasattr(g, 'adj') and g.adj is not None:
            adj = g.adj.float()
        else:
            adj = torch.zeros((n_roi, n_roi))
            ei = g.edge_index
            ea = g.edge_attr.view(-1) if g.edge_attr.dim() == 1 else g.edge_attr[:, 0]
            adj[ei[0], ei[1]] = ea
            adj[ei[1], ei[0]] = ea
        acc += adj
        count += 1
    if count == 0:
        return acc
    return acc / count


def convert_subject_arrays_to_reggnn(train_array, val_array, test_array,
                                      output_folder='./simulated_data/'):
    """
    Build subject-level mean connectomes used for SAMPLE SELECTION ONLY.
    Each subject -> one [ROI, ROI] mean connectome (averaged across windows).

    Order: train subjects, then val, then test.
    Returns: connectomes [ROI, ROI, n_subjects], scores [n_subjects], counts.
    """
    # train_array shape = (N_subj_train, T_win)  -- numpy 2D array of Data
    n_tr = train_array.shape[0]
    n_va = val_array.shape[0]
    n_te = test_array.shape[0]
    n_total = n_tr + n_va + n_te

    # Infer ROI from first non-padded window
    first_g = train_array[0, 0]
    n_roi = first_g.x.shape[0] if hasattr(first_g, 'x') else first_g.num_nodes

    print(f"  [Convert] subjects={n_total} (train={n_tr} val={n_va} test={n_te})  ROI={n_roi}")

    connectomes = torch.zeros((n_roi, n_roi, n_total))
    scores      = torch.zeros(n_total)

    def _y(g):
        if hasattr(g, 'y') and g.y is not None:
            return float(g.y.item() if hasattr(g.y, 'item') else g.y)
        return 0.0

    idx = 0
    for arr, label in [(train_array, 'train'), (val_array, 'val'), (test_array, 'test')]:
        for s in range(arr.shape[0]):
            connectomes[:, :, idx] = _subject_mean_adjacency(arr[s], n_roi)
            scores[idx]            = _y(arr[s, 0])  # score is constant across windows
            idx += 1

    os.makedirs(output_folder, exist_ok=True)
    torch.save(connectomes, f"{output_folder}connectome.ts")
    torch.save(scores,      f"{output_folder}score.ts")
    return connectomes, scores, n_tr, n_va, n_te


def create_pairwise_features_abs(connectomes_np, scores_np, verbose=True):
    """Pairwise topological features on subject-level mean connectomes."""
    n = connectomes_np.shape[2]
    data_dict, score_dict = {}, {}
    if verbose:
        print(f"    Building pairwise dicts: {n} subjects ({n*(n-1)//2} pairs) ...")
    for i in range(n):
        for j in range(i + 1, n):
            diff = np.abs(connectomes_np[:,:,i] - connectomes_np[:,:,j])
            feat = diff[np.triu_indices_from(diff, k=1)]
            data_dict[(i,j)]  = data_dict[(j,i)]  = feat
            score_dict[(i,j)] = score_dict[(j,i)] = float(np.abs(scores_np[i]-scores_np[j]))
    if verbose:
        print(f"    Pairwise dicts built ✓  ({len(data_dict)} entries)")
    return data_dict, score_dict


print("convert_subject_arrays_to_reggnn  defined ✓  (mean connectome per subject)")
print("create_pairwise_features_abs     defined ✓")


convert_subject_arrays_to_reggnn  defined ✓  (mean connectome per subject)
create_pairwise_features_abs     defined ✓


## Step 5: Configuration

`PRIMARY_METRIC` drives best-k selection and fine-tune early stopping.  
`K_FOLDS` is **not used** here — the 5 folds come from the 5 inner PKL files directly.

In [ ]:
# ─── Configuration ───────────────────────────────────────────────────────────
# Choices: 'mae' | 'rmse' | 'pearson' | 'spearman' | 'r' | 'r2'
PRIMARY_METRIC = 'pearson'

class Config:
    DATA_FOLDER    = './simulated_data/'   # overwritten per inner fold
    RESULT_FOLDER  = './results/'          # overwritten per inner fold
    ROI            = None                  # set per inner fold
    SPD            = True
    # K_FOLDS NOT used — folds come from the 5 inner PKL files
    SHUFFLE        = True
    DATA_SEED      = 1
    MODEL_SEED     = 1
    PRIMARY_METRIC = PRIMARY_METRIC

    class RegGNN:
        NUM_EPOCH = 100
        LR        = 1e-3
        WD        = 5e-4
        DROPOUT   = 0.1

    class SampleSelection:
        SAMPLE_SELECTION = True
        K_LIST           = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
        N_SELECT_SPLITS  = 10

os.makedirs(Config.RESULT_FOLDER, exist_ok=True)
print(f"PRIMARY_METRIC   : {PRIMARY_METRIC}")
print(f"Epochs           : {Config.RegGNN.NUM_EPOCH}")
print(f"Sample selection : {Config.SampleSelection.SAMPLE_SELECTION}")
print(f"K list           : {Config.SampleSelection.K_LIST}")


PRIMARY_METRIC   : pearson
Epochs           : 100
Sample selection : True
K list           : [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


## Step 6: Pairwise Feature Function Already Defined in Step 4

Pairwise features are built **per inner fold** inside the main loop.  
No global build needed here.

In [ ]:
# Pairwise features are built once per inner fold inside the Step 7 loop.
print("Pairwise feature extraction ready — called inside Step 7 per inner fold.")


Pairwise feature extraction ready — called inside Step 7 per inner fold.


### Step 7 — Improved trainer (v2)

This cell is an upgraded version of the original Step 7 with the following changes (see `analysis_and_fixes.md`):

1. Targets z-scored on train (predictions un-z-scored for metrics).
2. Final-layer bias initialised to zero (= `mean(y_train)` after z-score).
3. Node features = ROI rows of the adjacency, not identity.
4. Adjacency sanitised to `|A|` with diagonal=0 before message passing.
5. ROI readout = mean-pool (replaces degenerate `Linear(n_roi, 1)`).
6. Temporal aggregator = single-query attention (replaces 1-layer LSTM).
7. LayerNorm after the GNN encoder and after the temporal pool.
8. Loss = SmoothL1 (Huber) instead of MSE.
9. Phase 2 picks the **val-best epoch** (true early stopping) — not the last.
10. K-list floored at half the train size (avoids degenerate k=2 selections).
11. AdamW + cosine LR + 10-epoch warmup; 200 epochs with patience-30.
12. Gradient clipping at norm 1.0.

### Step 7 — Definitions only (TRegGNNv2, helpers, fit_and_eval)

Defines model, dataset, training utilities. **No training is performed here.** Step 7b (v16 runner) does the actual training.

`fit_and_eval` has the early-stop *break* commented out — val-best checkpointing is preserved, but training runs the full epoch budget.

In [ ]:
# ─── Improved T-RegGNN — All 5 Inner Folds ───────────────────────────────────
# Drop-in replacement for Step 7 (Cell 15) of RegGNN_Colab_v11.ipynb.
#
# Changes vs. original (see analysis_and_fixes.md for full rationale):
#   1. Targets z-scored on train; predictions un-z-scored for metrics.
#   2. Head bias initialised to mean(y_train) (= 0 after standardization).
#   3. Node features = ROI rows of the adjacency (subject-specific), not identity.
#   4. Adjacency |·| with diagonal zeroed before DenseGCNConv.
#   5. ROI readout = mean-pool (replaces degenerate Linear(n_roi,1)).
#   6. Temporal aggregator = single-query attention (replaces 1-layer LSTM).
#   7. LayerNorm after GNN encode and after temporal pool.
#   8. SmoothL1 (Huber) loss instead of MSE.
#   9. Val-best checkpointing in Phase-2; test metrics use the best-val epoch.
#  10. K_LIST floor — sample-selection keeps at least half the train subjects.
#  11. AdamW + cosine LR + 10-epoch warmup; early stop with patience.
#  12. Gradient clipping at norm 1.0.
# ─────────────────────────────────────────────────────────────────────────────

import os, pickle, random, csv, math
import numpy as np
from collections import defaultdict
from copy import deepcopy
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
import torch, torch.nn as nn, torch.nn.functional as F
import torch_geometric
from torch_geometric.nn.dense import DenseGCNConv
from scipy import stats


# ─────────────────────────────────────────────────────────────────────────────
# Improved T-RegGNN: connectivity-row features + GCN + temporal attention
# ─────────────────────────────────────────────────────────────────────────────
class TRegGNNv2(nn.Module):
    """
    Per-window GNN encoder followed by temporal-attention readout over windows.

    Inputs
    ------
    x_seq   : [B, T, ROI, ROI]   per-window node features (= adjacency rows)
    adj_seq : [B, T, ROI, ROI]   per-window adjacency for message passing
    lengths : [B]                real (non-padded) window count per subject
    """
    def __init__(self, n_roi, gnn_hidden=32, attn_hidden=32, dropout=0.2,
                 init_bias=0.0):
        super().__init__()
        self.n_roi = n_roi
        self.dropout = dropout

        # Per-window GNN encoder (2 GCN layers with residual + LayerNorm)
        self.gc1 = DenseGCNConv(n_roi, gnn_hidden)
        self.gc2 = DenseGCNConv(gnn_hidden, gnn_hidden)
        self.gn_ln = nn.LayerNorm(gnn_hidden)

        # Temporal attention readout over windows (single learned query)
        self.attn_proj  = nn.Linear(gnn_hidden, attn_hidden)
        self.attn_query = nn.Parameter(torch.randn(attn_hidden) * 0.02)
        self.t_ln       = nn.LayerNorm(gnn_hidden)

        # Regression head — bias initialised to train-mean of y (= 0 after z-score)
        self.head = nn.Linear(gnn_hidden, 1)
        with torch.no_grad():
            self.head.bias.fill_(float(init_bias))

    # ─── per-window encode → window embedding ────────────────────────────────
    def encode_window(self, x_w, adj_w):
        # Both [B, ROI, ROI]; mean-pool ROIs to a window vector.
        h = F.relu(self.gc1(x_w, adj_w))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.gc2(h, adj_w)                 # [B, ROI, gnn_hidden]
        h = h.mean(dim=1)                      # [B, gnn_hidden]  ← mean readout
        h = self.gn_ln(h)
        return h

    # ─── temporal attention over windows ─────────────────────────────────────
    def temporal_attention(self, emb_seq, lengths):
        # emb_seq: [B, T, gnn_hidden]; lengths: [B]
        B, T, D = emb_seq.shape
        # Mask padded positions
        mask = (torch.arange(T, device=emb_seq.device)[None, :] <
                lengths.to(emb_seq.device)[:, None])      # [B, T]  bool
        # Score = projected emb · learnable query
        proj   = torch.tanh(self.attn_proj(emb_seq))      # [B, T, attn_hidden]
        scores = (proj * self.attn_query).sum(-1)         # [B, T]
        scores = scores.masked_fill(~mask, float('-inf'))
        weights = F.softmax(scores, dim=1).unsqueeze(-1)  # [B, T, 1]
        pooled  = (emb_seq * weights).sum(dim=1)          # [B, gnn_hidden]
        return self.t_ln(pooled)

    def forward(self, x_seq, adj_seq, lengths):
        B, T = x_seq.shape[0], x_seq.shape[1]
        x_flat   = x_seq.reshape(B * T, self.n_roi, self.n_roi)
        adj_flat = adj_seq.reshape(B * T, self.n_roi, self.n_roi)
        emb_flat = self.encode_window(x_flat, adj_flat)
        emb_seq  = emb_flat.view(B, T, -1)
        subj_emb = self.temporal_attention(emb_seq, lengths)
        return self.head(subj_emb).view(-1)

    @staticmethod
    def loss(pred, target, beta=1.0):
        return F.smooth_l1_loss(pred.view(-1), target.view(-1), beta=beta)


# ─────────────────────────────────────────────────────────────────────────────
# Subject-sequence dataset — builds [T,ROI,ROI] features and adjacencies
# ─────────────────────────────────────────────────────────────────────────────
class SubjectSequenceDatasetV2(torch.utils.data.Dataset):
    """
    For each subject we precompute:
      x_seq[t]   = sanitized adjacency  (|·|, diagonal=0)  ← used as node features
      adj_seq[t] = same matrix          ← used for message passing
    Targets are stored in their *raw* scale; Trainer applies z-score.
    """
    def __init__(self, subj_array, n_roi):
        self.n_roi = n_roi
        self.subjects, self.scores, self.lengths = [], [], []
        for s in range(subj_array.shape[0]):
            T = subj_array.shape[1]
            x_seq   = torch.zeros(T, n_roi, n_roi)
            adj_seq = torch.zeros(T, n_roi, n_roi)
            real = 0
            for t in range(T):
                g = subj_array[s, t]
                if getattr(g, 'pad', False):
                    break
                if hasattr(g, 'adj') and g.adj is not None:
                    A = g.adj.float()
                else:
                    A = torch.zeros(n_roi, n_roi)
                    ei = g.edge_index
                    ea = g.edge_attr.view(-1) if g.edge_attr.dim() == 1 \
                         else g.edge_attr[:, 0]
                    A[ei[0], ei[1]] = ea
                    A[ei[1], ei[0]] = ea
                # Sanitize adjacency for DenseGCNConv: |A|, zero diagonal
                A = A.abs()
                A.fill_diagonal_(0.0)
                # Use the connectivity row as the node feature for that ROI
                x_seq[t]   = A
                adj_seq[t] = A
                real += 1
            if real == 0:
                continue
            y0 = subj_array[s, 0].y
            score = float(y0.item() if hasattr(y0, 'item') else y0)
            self.subjects.append((x_seq, adj_seq))
            self.scores.append(score)
            self.lengths.append(real)

    def __len__(self): return len(self.subjects)
    def __getitem__(self, i):
        x_seq, adj_seq = self.subjects[i]
        return x_seq, adj_seq, self.lengths[i], self.scores[i]


def subject_collate(batch):
    x_list, adj_list, lens, ys = zip(*batch)
    return (torch.stack(x_list, 0),
            torch.stack(adj_list, 0),
            torch.tensor(lens, dtype=torch.long),
            torch.tensor(ys,   dtype=torch.float))


def make_loader(ds, batch_size, shuffle):
    return torch.utils.data.DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, collate_fn=subject_collate)


# ─────────────────────────────────────────────────────────────────────────────
# Sample selection (unchanged math, but kept available; we floor K to half)
# ─────────────────────────────────────────────────────────────────────────────
def select_samples(train_idx, n_splits, k_list, dd, sd, shuffle, rs):
    freq = {k: defaultdict(int) for k in k_list}
    train_idx = np.array(train_idx)
    for tr_ids, ho_ids in KFold(n_splits, shuffle=shuffle,
                                random_state=rs).split(train_idx):
        res, serr = [], []
        for i in range(len(tr_ids)):
            for j in range(i + 1, len(tr_ids)):
                a, b = train_idx[tr_ids[i]], train_idx[tr_ids[j]]
                res.append(dd[(a, b)]); serr.append(sd[(a, b)])
        X = np.array(res); y = np.array(serr)
        if X.ndim == 1: X = X.reshape(-1, 1)
        reg = LinearRegression().fit(X, y)
        for hid in ho_ids:
            subj = train_idx[hid]
            Xt = np.array([dd[(subj, train_idx[t])] for t in tr_ids])
            if Xt.ndim == 1: Xt = Xt.reshape(-1, 1)
            ranked = train_idx[tr_ids[np.argsort(reg.predict(Xt).ravel())]]
            for k in k_list:
                for sid in ranked[:k]: freq[k][int(sid)] += 1
    out = {}
    for k in k_list:
        items = list(freq[k].items())
        if not items: out[k] = train_idx[:k]; continue
        ids = np.array([x[0] for x in items])
        fqs = np.array([x[1] for x in items])
        out[k] = ids[np.argsort(fqs)[::-1][:k]]
    return out


# ─────────────────────────────────────────────────────────────────────────────
# Metrics
# ─────────────────────────────────────────────────────────────────────────────
mae_ev  = lambda p, s: float(np.mean(np.abs(p - s)))
rmse_ev = lambda p, s: float(np.sqrt(np.mean((p - s) ** 2)))
prs_ev  = lambda p, s: float(stats.pearsonr(p, s)[0])  if len(p) > 1 else float('nan')
spr_ev  = lambda p, s: float(stats.spearmanr(p, s)[0]) if len(p) > 1 else float('nan')
r2_ev   = lambda p, s: float(1 - np.sum((s-p)**2) /
                              (np.sum((s-np.mean(s))**2) + 1e-12))
def r_ev(p, s):
    r2 = r2_ev(p, s); pr = prs_ev(p, s)
    return float(np.sqrt(max(0., r2))) * (1. if pr >= 0 else -1.)
def all_metrics(p, s):
    return {'mae': mae_ev(p, s), 'rmse': rmse_ev(p, s),
            'pearson': prs_ev(p, s), 'spearman': spr_ev(p, s),
            'r': r_ev(p, s), 'r2': r2_ev(p, s)}

_METRIC_FNS = {
    'mae': (mae_ev, False), 'rmse': (rmse_ev, False), 'pearson': (prs_ev, True),
    'spearman': (spr_ev, True), 'r': (r_ev, True), 'r2': (r2_ev, True),
}


# ─────────────────────────────────────────────────────────────────────────────
# Train / eval helpers — operate in standardized-y space
# ─────────────────────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, device, y_mean, y_std,
              train_mode=True, clip_norm=1.0):
    model.train(train_mode)
    preds_raw, ys_raw, losses = [], [], []
    grad_ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with grad_ctx:
        for x_seq, adj_seq, lengths, targets in loader:
            x_seq, adj_seq = x_seq.to(device), adj_seq.to(device)
            t_z = ((targets - y_mean) / y_std).to(device)
            out_z = model(x_seq, adj_seq, lengths)
            loss  = model.loss(out_z, t_z)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
                optimizer.step()
            # Un-z-score for reporting
            out_raw = out_z.detach().cpu().numpy() * y_std + y_mean
            preds_raw.append(out_raw)
            ys_raw.append(targets.numpy())
            losses.append(loss.item())
    return np.hstack(preds_raw).ravel(), np.hstack(ys_raw).ravel(), losses


def cosine_lr(opt, base_lr, epoch, total, warmup=10):
    if epoch < warmup:
        lr = base_lr * (epoch + 1) / max(1, warmup)
    else:
        prog = (epoch - warmup) / max(1, total - warmup)
        lr = 0.5 * base_lr * (1 + math.cos(math.pi * prog))
    for g in opt.param_groups: g['lr'] = lr


def fit_and_eval(model, train_loader, eval_loader, device, y_mean, y_std,
                 base_lr=1e-3, weight_decay=1e-3, epochs=200,
                 patience=30, primary='mae', primary_higher=False,
                 verbose_every=20, prefix=''):
    """Train with cosine LR + early stop on `primary` metric on eval_loader."""
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr,
                            weight_decay=weight_decay)
    best_score = -np.inf if primary_higher else np.inf
    best_state = deepcopy(model.state_dict())
    best_epoch = 0
    no_improve = 0

    for ep in range(epochs):
        cosine_lr(opt, base_lr, ep, epochs, warmup=10)
        tp, ty, tl = run_epoch(model, train_loader, opt, device, y_mean, y_std,
                                train_mode=True)
        vp, vy, _  = run_epoch(model, eval_loader,  opt, device, y_mean, y_std,
                                train_mode=False)
        vm = all_metrics(vp, vy)[primary]
        improved = (vm > best_score) if primary_higher else (vm < best_score)
        if improved and not (isinstance(vm, float) and (math.isnan(vm) or math.isinf(vm))):
            best_score, best_state = vm, deepcopy(model.state_dict())
            best_epoch, no_improve = ep + 1, 0
        else:
            no_improve += 1

        if (ep + 1) % verbose_every == 0 or ep == 0:
            tm = all_metrics(tp, ty)
            print(f"    {prefix} Ep {ep+1:3d}/{epochs}  "
                  f"loss={np.mean(tl):.4f}  "
                  f"tr MAE={tm['mae']:.3f} r={tm['pearson']:.3f}  |  "
                  f"val {primary}={vm:.4f}  "
                  f"(best {primary}={best_score:.4f} @ ep{best_epoch})")

        # ── Early-stop trigger DISABLED ─────────────────────────────────────
        # On tiny val sets (n=8-10 subjects) the val Pearson is noisy enough that
        # patience-based stopping fires on noise (cf. v13 'best @ ep1' pathology).
        # We keep best_state tracking so we still return the val-best checkpoint,
        # but train through the full epoch budget. To re-enable, uncomment below.
        #
        # if no_improve >= patience:
        #     print(f"    {prefix} Early stop @ ep{ep+1}  "
        #           f"(best {primary}={best_score:.4f} @ ep{best_epoch})")
        #     break

    model.load_state_dict(best_state)
    return model, best_score, best_epoch


# ═════════════════════════════════════════════════════════════════════════════
# Step 7 — DEFINITIONS ONLY
# ═════════════════════════════════════════════════════════════════════════════
# Step 7 used to also run a 5-inner-fold training loop here for OUTER_FOLD only,
# which is now redundant: Step 7b runs all 25 folds. We keep the helpers and
# class definitions above, set globals downstream cells need, and nothing else.

import os, torch
_device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_DIR = './results/'
os.makedirs(BASE_DIR, exist_ok=True)
print('Step 7 definitions loaded.  Device:', _device)
print('Run Step 7b to actually train (5 outer × 5 inner = 25 folds).')

# ── Backward-compat aliases for any code that still uses original names ─────
# (Step 8 / Step 9 in the original notebook reference these.)
SubjectSequenceDataset = SubjectSequenceDatasetV2
TRegGNN                = TRegGNNv2
def get_subject_loaders(train_ds, eval_ds, batch_size=4):
    return (make_loader(train_ds, batch_size, shuffle=True),
            make_loader(eval_ds,  batch_size, shuffle=False))
def run_one_epoch(model, loader, optimizer, device, train_mode=True):
    """Compat wrapper that uses raw-scale targets (no z-score)."""
    model.train(train_mode)
    preds, ys, losses = [], [], []
    grad_ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with grad_ctx:
        for x_seq, adj_seq, lengths, targets in loader:
            x_seq, adj_seq, targets = x_seq.to(device), adj_seq.to(device), targets.t

Step 7 definitions loaded.  Device: cuda
Run Step 7b to actually train (5 outer × 5 inner = 25 folds).


### Step 7b — v17 runner (anti-overfit)

v16 collapsed to **avg test r = -0.14** because the model memorised all 32 training subjects perfectly (train Pearson 0.9999) while val stayed noisy and there was no early stopping. v17 fixes:

- **Tiny model**: gnn/attn hidden 32 → 8 (8× fewer params)
- **Heavy regularisation**: dropout 0.20 → 0.50, weight-decay 10×
- **Early stop on val MAE** (stable on n=8) with patience 15
- **LR 5e-4**, 100 epoch budget (was 200)
- **Graph-noise augmentation**: ±5% Gaussian on adjacency at train time
- 3-seed ensemble, no per-fold calibration

In [ ]:
"""
v17 — 25-fold runner with AGGRESSIVE anti-overfit measures.

v16 trained to perfect train Pearson (0.9999) in ~25 epochs while val/test
fell apart (avg test r = -0.14). Root cause: model too large for n=32 train
subjects, val Pearson too noisy to checkpoint reliably, no early stopping.

v17 changes (in order of expected impact):
  A. Tiny model: gnn_hidden 32→8, attn_hidden 32→8.        (8x fewer params)
  B. Heavy dropout 0.50, weight_decay 1e-2 (10x v16).        (real reg)
  C. Early-stop on val MAE (stable on n=8), patience 15.    (no noise stop)
  D. Lower LR 5e-4 and 100 epoch budget.                    (no overshoot)
  E. Graph-noise augmentation: random Gaussian on adjacency at train time.
  F. 3-seed ensemble + per-run Drive folder + one consolidated JSON.
"""

import os, csv, time, random, math, datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from sklearn.linear_model import LinearRegression as _LR

# Restore SmoothL1-only loss
TRegGNNv2.loss = staticmethod(
    lambda p, t, beta=1.0: F.smooth_l1_loss(p.view(-1), t.view(-1), beta=beta))

# Remove any v16 attention temperature (was destabilising)
if hasattr(TRegGNNv2, 'attn_temperature'):
    delattr(TRegGNNv2, 'attn_temperature')

# ── Hyper-parameters ───────────────────────────────────────────────────────
BATCH_SIZE  = 4
GNN_H       = 8         # was 32 — drastically smaller
ATTN_H      = 8         # was 32
DROP        = 0.50      # was 0.20
EPOCHS_25   = 100       # was 200
PATIENCE_25 = 15
LR_25       = 5e-4      # was 1e-3
WD_25       = 1e-2      # was 5e-3 (now 10x v16)
NOISE_SIGMA = 0.05      # train-time Gaussian noise on adjacency
N_SEEDS_25  = 3
SEED_BASE   = Config.MODEL_SEED
N_OUTER     = 5
PRINT_EVERY = 10

# ── Step-7-style print helper (unchanged) ───────────────────────────────────
def print_epoch_v7(prefix, ep, total, p, s, losses):
    p, s = np.asarray(p, dtype=float), np.asarray(s, dtype=float)
    mae  = float(np.mean(np.abs(p - s)))
    rmse = float(np.sqrt(np.mean((p - s) ** 2)))
    mape = float(np.mean(np.abs((p - s) / (np.abs(s) + 1e-8))) * 100)
    prs  = prs_ev(p, s); spr = spr_ev(p, s)
    ss_r = float(np.sum((s - p)**2))
    ss_t = float(np.sum((s - np.mean(s))**2)) + 1e-12
    r2   = float(1 - ss_r / ss_t)
    loss = float(np.mean(losses))
    print(f"    {prefix} Epoch [{ep:3d}/{total}]  "
          f"Loss(MSE):{loss:.4f}  MAE:{mae:.4f}  RMSE:{rmse:.4f}  "
          f"MAPE:{mape:.2f}%  Pearson:{prs:.4f}  Spearman:{spr:.4f}  R2:{r2:.4f}")


# ── Graph-noise augmentation wrapper for run_epoch ─────────────────────────
def run_epoch_aug(model, loader, optimizer, device, y_mean, y_std,
                   train_mode=True, noise_sigma=0.0, grad_clip=1.0):
    model.train(train_mode)
    preds_raw, ys_raw, losses = [], [], []
    grad_ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with grad_ctx:
        for x_seq, adj_seq, lengths, targets in loader:
            x_seq, adj_seq = x_seq.to(device), adj_seq.to(device)
            t_z = ((targets - y_mean) / y_std).to(device)
            if train_mode and noise_sigma > 0:
                # Add Gaussian noise to adjacency at training time only
                noise = torch.randn_like(adj_seq) * noise_sigma
                # symmetric noise + zero diagonal
                noise = (noise + noise.transpose(-1, -2)) * 0.5
                diag = torch.eye(adj_seq.shape[-1], device=device).bool()
                noise = noise.masked_fill(diag, 0.0)
                adj_in = (adj_seq + noise).clamp_min(0.0)
                x_in   = adj_in
            else:
                adj_in = adj_seq; x_in = x_seq
            out_z = model(x_in, adj_in, lengths)
            loss  = model.loss(out_z, t_z)
            if train_mode:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            out_raw = out_z.detach().cpu().numpy() * y_std + y_mean
            preds_raw.append(out_raw)
            ys_raw.append(targets.numpy())
            losses.append(loss.item())
    return np.hstack(preds_raw).ravel(), np.hstack(ys_raw).ravel(), losses


def train_v17(model, tr_loader, va_loader, device, y_mean, y_std,
               base_lr, weight_decay, epochs, patience,
               prefix='[v17]', print_every=PRINT_EVERY):
    """Train with early stop on val MAE (stable on n=8). Restore val-best ckpt."""
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr,
                            weight_decay=weight_decay)
    best_mae = float('inf'); best_state = deepcopy(model.state_dict()); best_ep = 0
    no_improve = 0
    for ep in range(epochs):
        cosine_lr(opt, base_lr, ep, epochs, warmup=5)
        tp, ty, tl = run_epoch_aug(model, tr_loader, opt, device,
                                    y_mean, y_std, train_mode=True,
                                    noise_sigma=NOISE_SIGMA)
        vp, vy, _  = run_epoch_aug(model, va_loader, opt, device,
                                    y_mean, y_std, train_mode=False)
        vmae = float(np.mean(np.abs(vp - vy)))
        if vmae < best_mae - 1e-4:
            best_mae = vmae
            best_state = deepcopy(model.state_dict())
            best_ep = ep + 1
            no_improve = 0
        else:
            no_improve += 1
        if (ep + 1) % print_every == 0 or ep == 0:
            print_epoch_v7(prefix + ' Train', ep + 1, epochs, tp, ty, tl)
            print_epoch_v7(prefix + ' Val  ', ep + 1, epochs, vp, vy, [0.0])
            print(f"      (best val MAE={best_mae:.4f} @ ep{best_ep})")
        if no_improve >= patience:
            print(f"    {prefix} Early stop @ ep{ep+1}  "
                  f"(best val MAE={best_mae:.4f} @ ep{best_ep})")
            break
    model.load_state_dict(best_state)
    return model, best_mae, best_ep


# ── Outer × Inner runner ──────────────────────────────────────────────────
t0 = time.time()
print('#' * 78)
print('  v17 — 25-fold runner (anti-overfit)')
print('  GNN_H=%d  ATTN_H=%d  DROPOUT=%.2f  WD=%.1e  LR=%.1e' %
      (GNN_H, ATTN_H, DROP, WD_25, LR_25))
print('  Early-stop on val MAE  patience=%d  noise=%.2f' %
      (PATIENCE_25, NOISE_SIGMA))
print('#' * 78)

all_results, agg_subj_preds_25, agg_subj_ys_25 = {}, [], []
agg_subj_outer_25, agg_subj_inner_25 = [], []
best_models_per_fold, best_model_meta = {}, {}

def _all_inner_pkls(outer):
    paths = {}
    for inner in range(1, 5 + 1):
        fname = 'graphs_outer%d_inner%d.pkl' % (outer, inner)
        path  = os.path.join(FOLDS_DATA_DIR, fname)
        if not os.path.exists(path): raise FileNotFoundError(path)
        paths[inner] = path
    return paths

for outer in range(1, N_OUTER + 1):
    print('\n' + '#' * 78)
    print('  OUTER FOLD %d / %d' % (outer, N_OUTER))
    print('#' * 78)
    inner_pkls_o = _all_inner_pkls(outer)
    all_results[outer] = {}

    for inner in range(1, 5 + 1):
        print('\n' + '=' * 70)
        print('  INNER FOLD %d/5 (outer %d)  |  graphs_outer%d_inner%d.pkl' %
              (inner, outer, outer, inner))
        print('=' * 70)
        pkl_data = torch.load(inner_pkls_o[inner], map_location='cpu',
                               weights_only=False)
        train_arr = pkl_data['train_graphs']
        val_arr   = pkl_data['val_graphs']
        test_arr  = pkl_data['test_graphs']
        first_g   = train_arr[0, 0]
        n_roi     = first_g.x.shape[0] if hasattr(first_g, 'x') else first_g.num_nodes
        Config.ROI = int(n_roi)

        train_ds = SubjectSequenceDatasetV2(train_arr, n_roi)
        val_ds   = SubjectSequenceDatasetV2(val_arr,   n_roi)
        test_ds  = SubjectSequenceDatasetV2(test_arr,  n_roi)
        y_train  = np.array(train_ds.scores, dtype=np.float32)
        y_mean   = float(y_train.mean()); y_std = float(y_train.std() + 1e-8)
        print('  [Datasets] train=%d val=%d test=%d  ROI=%d  T=%d' %
              (len(train_ds), len(val_ds), len(test_ds), n_roi, train_arr.shape[1]))
        print('  [Targets ] mean=%.3f std=%.3f' % (y_mean, y_std))

        val_loader  = make_loader(val_ds,  BATCH_SIZE, shuffle=False)
        test_loader = make_loader(test_ds, BATCH_SIZE, shuffle=False)

        test_preds_seeds = []
        first_model = None; test_ys_loc = None
        for s_idx in range(N_SEEDS_25):
            seed = SEED_BASE + 17 * s_idx + 101 * outer + 7 * inner
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
            print('\n  --- seed %d / %d (outer %d  inner %d) ---' %
                  (s_idx + 1, N_SEEDS_25, outer, inner))
            tr_loader = make_loader(train_ds, BATCH_SIZE, shuffle=True)
            model = TRegGNNv2(n_roi, gnn_hidden=GNN_H, attn_hidden=ATTN_H,
                               dropout=DROP, init_bias=0.0).to(_device)
            model, _, best_ep = train_v17(
                model, tr_loader, val_loader, _device, y_mean, y_std,
                base_lr=LR_25, weight_decay=WD_25,
                epochs=EPOCHS_25, patience=PATIENCE_25,
                prefix='[o%d i%d s%d]' % (outer, inner, s_idx))
            tp, ty, _ = run_epoch_aug(model, test_loader, None, _device,
                                        y_mean, y_std, train_mode=False)
            test_preds_seeds.append(tp); test_ys_loc = ty
            if first_model is None: first_model = model

        test_pred = np.mean(np.stack(test_preds_seeds, 0), axis=0)
        m  = all_metrics(test_pred, test_ys_loc)
        mp = float(np.mean(np.abs((test_pred - test_ys_loc) /
                                    (np.abs(test_ys_loc) + 1e-8))) * 100)

        print('\n  >> Test (ensemble %d): MAE=%.4f RMSE=%.4f MAPE=%.2f%%' %
              (N_SEEDS_25, m['mae'], m['rmse'], mp))
        print('             Pearson r=%.4f  Spearman rho=%.4f  R=%.4f  '
              'R2=%.4f  [n=%d]' %
              (m['pearson'], m['spearman'], m['r'], m['r2'], len(test_ds)))

        all_results[outer][inner] = {
            'test_metrics': m, 'cal_a': 1.0, 'cal_b': 0.0,
            'n_test_subjects': len(test_ds), 'best_k': len(train_ds)}
        best_models_per_fold[(outer, inner)] = first_model
        best_model_meta[(outer, inner)] = {'y_mean': y_mean, 'y_std': y_std,
                                             'roi': n_roi}
        agg_subj_preds_25.extend(test_pred.tolist())
        agg_subj_ys_25.extend(test_ys_loc.tolist())
        agg_subj_outer_25.extend([outer] * len(test_ys_loc))
        agg_subj_inner_25.extend([inner] * len(test_ys_loc))
        print('  ✓ outer %d inner %d done  Pearson=%.4f' %
              (outer, inner, m['pearson']))

# ── Summary ────────────────────────────────────────────────────────────────
elapsed = time.time() - t0
flat = [(o,i,all_results[o][i]['test_metrics']) for o in range(1,N_OUTER+1)
        for i in range(1,5+1)]
prs = [m['pearson']  for _,_,m in flat]
spr = [m['spearman'] for _,_,m in flat]
mae = [m['mae']      for _,_,m in flat]
rms = [m['rmse']     for _,_,m in flat]
r2_ = [m['r2']       for _,_,m in flat]
agg_p = np.array(agg_subj_preds_25); agg_y = np.array(agg_subj_ys_25)
agg_m = all_metrics(agg_p, agg_y)

print('\n' + '*' * 78)
print('  v17 — 25-FOLD GRAND SUMMARY  (elapsed %.1f min)' % (elapsed/60.0))
print('*' * 78)
print('  outer  inner   Pearson   Spearman      MAE     RMSE       R2')
print('  ' + '-' * 60)
for (o,i,m) in flat:
    print('  %5d  %5d   %7.4f   %8.4f   %6.3f   %6.3f   %6.3f' %
          (o,i,m['pearson'],m['spearman'],m['mae'],m['rmse'],m['r2']))
print('  ' + '-' * 60)
print('  Avg over 25 folds : r=%.4f±%.4f  rho=%.4f  MAE=%.3f  RMSE=%.3f  R2=%.4f' %
      (np.nanmean(prs), np.nanstd(prs), np.nanmean(spr),
       np.nanmean(mae), np.nanmean(rms), np.nanmean(r2_)))
for o in range(1, N_OUTER+1):
    pso = [all_results[o][i]['test_metrics']['pearson'] for i in range(1,6)]
    print('   outer %d avg r = %.4f ± %.4f' %
          (o, np.nanmean(pso), np.nanstd(pso)))
print('  Aggregate (n=%d): r=%.4f rho=%.4f MAE=%.3f RMSE=%.3f R2=%.4f' %
      (len(agg_y), agg_m['pearson'], agg_m['spearman'],
       agg_m['mae'], agg_m['rmse'], agg_m['r2']))
print('*' * 78)

# Per-run Drive folder (training results go alongside XAI results)
import datetime as _dt
_RUN_TS = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
_DRIVE_BASE   = '/content/drive/MyDrive/GNN-mri/runs'
try:
    os.makedirs(_DRIVE_BASE, exist_ok=True)
    RUN_DIR = os.path.join(_DRIVE_BASE, 'run_17_2_' + _RUN_TS)
    os.makedirs(RUN_DIR, exist_ok=True)
except Exception:
    RUN_DIR = os.path.join(BASE_DIR, 'run_17_2_' + _RUN_TS)
    os.makedirs(RUN_DIR, exist_ok=True)
csv_path = os.path.join(RUN_DIR, 'summary_25folds.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['outer','inner','pearson','spearman','mae','rmse','r2','n_test'])
    for o in range(1, N_OUTER+1):
        for i in range(1, 5+1):
            r = all_results[o][i]; m = r['test_metrics']
            w.writerow([o,i,round(m['pearson'],6),round(m['spearman'],6),
                        round(m['mae'],6),round(m['rmse'],6),round(m['r2'],6),
                        r['n_test_subjects']])
print('Saved training summary:', csv_path)

# Compatibility exposes for downstream cells
inner_results = all_results[1]
agg_subj_preds, agg_subj_ys = agg_subj_preds_25, agg_subj_ys_25
test_preds, test_ys = agg_p, agg_y


##############################################################################
  v17 — 25-fold runner (anti-overfit)
  GNN_H=8  ATTN_H=8  DROPOUT=0.50  WD=1.0e-02  LR=5.0e-04
  Early-stop on val MAE  patience=15  noise=0.05
##############################################################################

##############################################################################
  OUTER FOLD 1 / 5
##############################################################################

  INNER FOLD 1/5 (outer 1)  |  graphs_outer1_inner1.pkl


## Step 8 — Explainability (single JSON, saved to Drive)

Triangulation: Integrated Gradients + temporal attention + ROI/window occlusion. Outputs go to **`/content/drive/MyDrive/fyp/runs/run_<TIMESTAMP>/`** — a new folder every execution.

**One consolidated `xai_results.json`** with sections:
- `metadata` — timestamp, n_roi, T_windows, n_folds, primary_metric
- `nodes` — every ROI: `roi, roi_name, network, x, y, z, node_strength, occlusion_importance`
- `edges` — every (i, j): `start_node, end_node, names, networks, node strengths, edge_importance, edge_type` (intra/inter-network)
- `top_rois` — top-30 ROIs by IG and occlusion
- `timeseries` — `T, ROI, overall_mean, per_roi`
- `attention_per_fold` — per-fold temporal-attention diagnostic

All PNG figures are copied into the same Drive folder.

In [ ]:
"""
v17 explainability — same IG + attention + occlusion analysis as v16, plus:

  Group level (across 25 folds):
    A. ROI ranking bar + temporal importance + IG-edge heatmap + per-fold
       attention heatmap (compact 2x2 panel — kept from v16).
    B. NEW: 2-D brain plot — three anatomical views (axial / sagittal /
       coronal) at Shen-268 MNI coordinates. Node size+colour = IG ROI
       importance, edges = top-K IG edge importance, brain outline drawn
       semi-transparent.
    C. NEW: 3-D brain plot — translucent ellipsoidal brain mesh; ROIs as 3-D
       scatter coloured by IG importance; top-K edges as 3-D line segments.
    D. NEW: Correlation-matrix figure — mean adjacency + IG edge importance.
    E. NEW: Time-series figure — overall ROI-mean per window + top-5 ROIs
       individually, with temporal attention curve aligned below.

  Subject level (one example subject):
    F. ROI x window IG heatmap + attention curve + top-5 connectivity time
       series for that subject.
    G. Predicted vs true scatter across all 250 test subjects.

Coordinates: Shen-268 MNI coordinates are loaded from `./shen268_coords.csv`
when present (three numeric columns: x, y, z in mm). Otherwise the code falls
back to a deterministic bilaterally-symmetric ellipsoid layout. To use real
Shen-268 coordinates, drop the CSV into the notebook working directory.
"""
import os, csv, math, json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.patches  import Ellipse
from mpl_toolkits.mplot3d.art3d import Line3DCollection


# ── Shen-268 ROI label / network loader ─────────────────────────────────
def get_shen268_labels(n_roi=268, csv_candidates=None):
    """
    Try to load Shen-268 ROI names + functional-network assignments from a CSV.

    Expected CSV columns (header row required): 'ROI', 'name', 'network'.
    Returns:
        roi_names    -- list[str] length n_roi (e.g. 'L_Precuneus' or 'ROI_42')
        roi_networks -- list[str] length n_roi (e.g. 'DMN', 'FPN', ...)

    Drop `shen268_labels.csv` into the working directory to use real labels.
    Otherwise falls back to: roi name = 'ROI_<i>',
                              network  = 'LH' for i<n/2 else 'RH'.
    """
    import glob
    if csv_candidates is None:
        csv_candidates = [
            './shen268_labels.csv',
            './shen_268_labels.csv',
            os.path.join(BASE_DIR, 'shen268_labels.csv'),
            '/content/shen268_labels.csv',
        ]
        # Auto-discover anything in /content/drive/MyDrive/GNN-mri/data/
        for pat in (
            '/content/drive/MyDrive/GNN-mri/data/shen_268_parcellation_networklabels.csv',
            '/content/drive/MyDrive/GNN-mri/data/*networklabels*.csv',
            '/content/drive/MyDrive/GNN-mri/data/*shen268*label*.csv',
            '/content/drive/MyDrive/GNN-mri/data/*parcellation*label*.csv',
            '/content/drive/MyDrive/GNN-mri/data/*label*.csv',
        ):
            csv_candidates.extend(sorted(glob.glob(pat)))

    # Standard Shen-268 functional-network names by integer ID (Finn et al. 2015)
    SHEN_NETS = {
        1: 'MedialFrontal',   2: 'Frontoparietal',
        3: 'DefaultMode',     4: 'SubcorticalCerebellum',
        5: 'Motor',           6: 'VisualI',
        7: 'VisualII',        8: 'VisualAssoc',
    }

    def _pick(row_ci, *keys):
        """Case-insensitive lookup across multiple key candidates."""
        for k in keys:
            v = row_ci.get(k.lower())
            if v is not None and v != '':
                return v
        return None

    names = ['ROI_%d' % i for i in range(n_roi)]
    nets  = ['LH' if i < n_roi // 2 else 'RH' for i in range(n_roi)]
    for p in csv_candidates:
        if not os.path.exists(p): continue
        try:
            with open(p) as f:
                reader = csv.DictReader(f)
                if reader.fieldnames is None: continue
                # Normalize keys to lowercase for case-insensitive access
                rows = [{k.strip().lower(): (v.strip() if isinstance(v, str) else v)
                          for k, v in r.items()} for r in reader]
            if not rows: continue
            # Detect whether ROI indices are 1-based (Shen standard) or 0-based
            # — scan ALL rows (Shen file has Node 1..268, so min=1 and max=268).
            all_idx = []
            for r in rows:
                v = _pick(r, 'ROI', 'roi', 'idx', 'node', 'Node', 'index')
                if v is not None:
                    try: all_idx.append(int(float(v)))
                    except Exception: pass
            one_based = (bool(all_idx) and min(all_idx) >= 1
                          and (max(all_idx) >= n_roi or 0 not in all_idx))
            offset = 1 if one_based else 0

            for r in rows:
                v = _pick(r, 'ROI', 'roi', 'idx', 'node', 'Node', 'index')
                if v is None: continue
                try: idx = int(float(v)) - offset
                except Exception: continue
                if idx < 0 or idx >= n_roi: continue
                nm = _pick(r, 'name', 'label', 'roi_name')
                nw = _pick(r, 'network', 'Network', 'net', 'networkid', 'network_id')
                if nm: names[idx] = str(nm)
                if nw is not None:
                    try:
                        ni = int(float(nw))
                        nets[idx] = SHEN_NETS.get(ni, 'Net_%d' % ni)
                    except Exception:
                        nets[idx] = str(nw)
            print('  [labels] loaded Shen-268 labels from', p,
                  ' (one-based ROI indexing? %s)' % one_based)
            return names, nets
        except Exception:
            continue
    print('  [labels] WARNING: no Shen-268 labels CSV found — using '
          "ROI_<i> + LH/RH FALLBACK. Place shen_268_parcellation_"
          "networklabels.csv (or similar) in /content/drive/MyDrive/GNN-mri/data/.")
    return names, nets

# Defensive: ensure BATCH_SIZE is defined
if 'BATCH_SIZE' not in globals():
    BATCH_SIZE = 4


# ── Patched forward returning attention weights ──────────────────────────
def forward_with_attn(self, x_seq, adj_seq, lengths):
    B, T = x_seq.shape[0], x_seq.shape[1]
    x_flat   = x_seq.reshape(B*T, self.n_roi, self.n_roi)
    adj_flat = adj_seq.reshape(B*T, self.n_roi, self.n_roi)
    emb_flat = self.encode_window(x_flat, adj_flat)
    emb_seq  = emb_flat.view(B, T, -1)
    proj   = torch.tanh(self.attn_proj(emb_seq))
    scores = (proj * self.attn_query).sum(-1)
    if hasattr(self, 'attn_temperature'):
        scores = scores * self.attn_temperature
    mask   = (torch.arange(T, device=emb_seq.device)[None, :] <
              lengths.to(emb_seq.device)[:, None])
    scores  = scores.masked_fill(~mask, float('-inf'))
    weights = F.softmax(scores, dim=1)
    pooled  = (emb_seq * weights.unsqueeze(-1)).sum(dim=1)
    pooled  = self.t_ln(pooled)
    out     = self.head(pooled).view(-1)
    return out, weights
TRegGNNv2.forward_with_attn = forward_with_attn


# ── Integrated Gradients & occlusion ─────────────────────────────────────
def integrated_gradients(model, adj_seq, lengths, baseline=None, steps=20):
    model.eval()
    if baseline is None: baseline = torch.zeros_like(adj_seq)
    grads = torch.zeros_like(adj_seq)
    for a in torch.linspace(0., 1., steps + 1, device=adj_seq.device):
        adj_a = baseline + a * (adj_seq - baseline)
        adj_a = adj_a.detach().requires_grad_(True)
        out = model(adj_a, adj_a, lengths)
        out.sum().backward()
        grads = grads + adj_a.grad.detach()
        adj_a.grad = None
    return ((adj_seq - baseline) * grads / float(steps + 1)).detach()

def window_occlusion(model, adj_seq, lengths):
    model.eval(); B, T = adj_seq.shape[0], adj_seq.shape[1]
    with torch.no_grad():
        base = model(adj_seq, adj_seq, lengths).detach().cpu().numpy()
        out = np.zeros((B, T), dtype=np.float32)
        for t in range(T):
            ap = adj_seq.clone(); ap[:, t] = 0
            out[:, t] = np.abs(model(ap, ap, lengths).detach().cpu().numpy() - base)
    return out

def roi_occlusion(model, adj_seq, lengths):
    model.eval(); B, T, R, _ = adj_seq.shape
    with torch.no_grad():
        base = model(adj_seq, adj_seq, lengths).detach().cpu().numpy()
        out = np.zeros((B, R), dtype=np.float32)
        for i in range(R):
            ap = adj_seq.clone(); ap[:, :, i, :] = 0; ap[:, :, :, i] = 0
            out[:, i] = np.abs(model(ap, ap, lengths).detach().cpu().numpy() - base)
    return out


# ── Shen-268 coordinate loader (with fallback) ──────────────────────────
def get_shen268_coords(n_roi=268, csv_candidates=None):
    """
    Try to load Shen-268 MNI coordinates from a CSV. Falls back to a
    deterministic bilateral ellipsoid layout.

    Searches (in order):
      1. ./shen268_coords.csv, ./shen_268_coords.csv
      2. <BASE_DIR>/shen268_coords.csv
      3. /content/shen268_coords.csv
      4. /content/drive/MyDrive/GNN-mri/data/*shen268*.csv  (user-uploaded)
      5. /content/drive/MyDrive/GNN-mri/data/shen*.csv

    Robust to several common formats:
      - 3 columns: x, y, z                          (used directly)
      - 4+ cols, first looks like ROI index 0..N-1  (cols 1-3 used)
      - 4+ cols, first does not look like an index  (cols 0-2 used)
      - Optional one-row header
    """
    import glob
    if csv_candidates is None:
        csv_candidates = [
            './shen268_coords.csv',
            './shen_268_coords.csv',
            os.path.join(BASE_DIR, 'shen268_coords.csv'),
            '/content/shen268_coords.csv',
        ]
        # Also auto-discover anything inside the user's Google Drive fyp/data
        for pat in ('/content/drive/MyDrive/GNN-mri/data/*shen268*.csv',
                     '/content/drive/MyDrive/GNN-mri/data/*shen_268*.csv',
                     '/content/drive/MyDrive/GNN-mri/data/shen*.csv'):
            csv_candidates.extend(sorted(glob.glob(pat)))

    def _extract_xyz(arr, path):
        """Pick the right 3 columns from a 2-D numeric array."""
        if arr.ndim != 2 or arr.shape[1] < 3: return None
        # If first column looks like a 0/1-based ROI index, drop it.
        c0 = arr[:, 0]
        looks_like_index = (
            arr.shape[1] >= 4
            and np.allclose(c0, np.round(c0))
            and (c0.min() in (0.0, 1.0))
            and (c0.max() <= arr.shape[0] + 1)
        )
        cols = (1, 2, 3) if looks_like_index else (0, 1, 2)
        if arr.shape[1] < max(cols) + 1: return None
        if arr.shape[0] < n_roi: return None
        print('  [coords] loaded Shen-268 coords from:', path,
              ' (cols %s, dropped index col? %s)' %
              (str(cols), looks_like_index))
        return arr[:n_roi][:, list(cols)].astype(np.float32)

    for p in csv_candidates:
        if not os.path.exists(p): continue
        for skip in (0, 1, 2):
            try:
                arr = np.loadtxt(p, delimiter=',', skiprows=skip)
            except Exception:
                # Try whitespace / tab delim as a fallback
                try:
                    arr = np.loadtxt(p, skiprows=skip)
                except Exception:
                    continue
            xyz = _extract_xyz(arr, p)
            if xyz is not None: return xyz

    print('  [coords] WARNING: no Shen-268 coords CSV found — using bilateral '
          'ellipsoid FALLBACK. Place shen268_coords.csv in working dir or in '
          '/content/drive/MyDrive/GNN-mri/data/ for real coordinates.')
    rng = np.random.default_rng(42)
    half = n_roi // 2
    a, b, c = 70.0, 85.0, 60.0   # approximate brain bounds in MNI mm
    phi   = rng.uniform(0.05, np.pi - 0.05, half)
    theta = rng.uniform(np.pi*0.55, np.pi*1.45, half)
    xs = a * np.sin(phi) * np.cos(theta)
    ys = b * np.sin(phi) * np.sin(theta) - 12.0
    zs = c * np.cos(phi) * 0.9 + 12.0
    left  = np.stack([xs, ys, zs], axis=1)
    right = left.copy(); right[:, 0] *= -1
    coords = np.concatenate([left, right], axis=0)
    if coords.shape[0] < n_roi:
        coords = np.concatenate(
            [coords, np.zeros((n_roi - coords.shape[0], 3), dtype=np.float32)],
            axis=0)
    return coords.astype(np.float32)


# ── 2-D brain (three anatomical views) ──────────────────────────────────
def _brain_outline(ax, plane, xlim, ylim, alpha=0.10):
    cx = (xlim[0] + xlim[1]) / 2; cy = (ylim[0] + ylim[1]) / 2
    if plane == 'axial':      ew, eh = 140, 175
    elif plane == 'sagittal': ew, eh = 175, 130
    else:                     ew, eh = 140, 130
    ax.add_patch(Ellipse((cx, cy), ew, eh, facecolor='#7777aa',
                          alpha=alpha, edgecolor='#444477', lw=1.0, zorder=1))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect('equal'); ax.axis('off')


def plot_brain_2d(fig, coords, roi_importance, edge_importance,
                   top_k_edges=120, top_label=15):
    R = coords.shape[0]
    iu, ju = np.triu_indices(R, k=1)
    ev = edge_importance[iu, ju]
    keep_n = min(top_k_edges, ev.size)
    sel = np.argsort(ev)[::-1][:keep_n]
    ev_sel  = ev[sel]
    ev_norm = (ev_sel - ev_sel.min()) / (ev_sel.max() - ev_sel.min() + 1e-12)

    imp_norm = roi_importance / (roi_importance.max() + 1e-12)
    sizes    = 12 + 110 * imp_norm

    axs = fig.subplots(1, 3)
    views = [
        ('axial',    0, 1, (-80,  80), (-110, 90)),
        ('sagittal', 1, 2, (-110, 90), (-65,  90)),
        ('coronal',  0, 2, (-80,  80), (-65,  90)),
    ]
    for ax, (name, i_dim, j_dim, xlim, ylim) in zip(axs, views):
        _brain_outline(ax, name, xlim, ylim)
        segs = [[(coords[iu[k], i_dim], coords[iu[k], j_dim]),
                 (coords[ju[k], i_dim], coords[ju[k], j_dim])] for k in sel]
        if len(segs):
            lc = LineCollection(segs, colors=plt.cm.viridis(ev_norm),
                                 linewidths=0.4 + 2.0 * ev_norm,
                                 alpha=0.15 + 0.55 * ev_norm)
            ax.add_collection(lc)
        ax.scatter(coords[:, i_dim], coords[:, j_dim],
                    s=sizes, c=imp_norm, cmap='magma',
                    edgecolor='black', linewidth=0.4, zorder=4)
        top = np.argsort(roi_importance)[::-1][:top_label]
        for r in top:
            ax.text(coords[r, i_dim] + 1.5, coords[r, j_dim] + 1.5,
                     str(r), fontsize=6, color='black',
                     bbox=dict(facecolor='white', alpha=0.55, lw=0, pad=0.5),
                     zorder=5)
        ax.set_title(name + ' view', fontsize=11)
    return axs


# ── 3-D brain (translucent ellipsoidal mesh + nodes + edges) ─────────────
def plot_brain_3d(ax, coords, roi_importance, edge_importance,
                   top_k_edges=80, top_label=15):
    R = coords.shape[0]
    u = np.linspace(0, 2 * np.pi, 30)
    v = np.linspace(0,     np.pi, 16)
    a, b, c = 75.0, 90.0, 65.0
    cx, cy, cz = 0.0, -8.0, 12.0
    X = cx + a * np.outer(np.cos(u), np.sin(v))
    Y = cy + b * np.outer(np.sin(u), np.sin(v))
    Z = cz + c * np.outer(np.ones_like(u), np.cos(v))
    ax.plot_surface(X, Y, Z, color='#8888bb', alpha=0.08, linewidth=0,
                     antialiased=True, shade=False)
    ax.plot_wireframe(X, Y, Z, color='#5555aa', alpha=0.15, linewidth=0.3)

    iu, ju = np.triu_indices(R, k=1)
    ev = edge_importance[iu, ju]
    keep_n = min(top_k_edges, ev.size)
    sel = np.argsort(ev)[::-1][:keep_n]
    if len(sel):
        ev_sel  = ev[sel]
        ev_norm = (ev_sel - ev_sel.min()) / (ev_sel.max() - ev_sel.min() + 1e-12)
        segs = [[(coords[iu[k], 0], coords[iu[k], 1], coords[iu[k], 2]),
                 (coords[ju[k], 0], coords[ju[k], 1], coords[ju[k], 2])]
                for k in sel]
        lc = Line3DCollection(segs, colors=plt.cm.viridis(ev_norm),
                               linewidths=0.5 + 2.5 * ev_norm,
                               alpha=0.20 + 0.55 * ev_norm)
        ax.add_collection3d(lc)

    imp_norm = roi_importance / (roi_importance.max() + 1e-12)
    sizes    = 18 + 200 * imp_norm
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                s=sizes, c=imp_norm, cmap='magma',
                edgecolor='black', linewidth=0.4, depthshade=True)

    top = np.argsort(roi_importance)[::-1][:top_label]
    for r in top:
        ax.text(coords[r, 0], coords[r, 1], coords[r, 2] + 3,
                 str(r), fontsize=6, color='black',
                 bbox=dict(facecolor='white', alpha=0.6, lw=0, pad=0.5))
    ax.set_xlabel('X (L↔R)')
    ax.set_ylabel('Y (P↔A)')
    ax.set_zlabel('Z (I↔S)')
    ax.set_title('3-D brain - node = ROI importance, edge = IG importance')
    ax.view_init(elev=18, azim=-60)


# ═════════════════════════════════════════════════════════════════════════
# Run XAI across all 25 folds
# ═════════════════════════════════════════════════════════════════════════
print('=' * 78)
print('  v17 EXPLAINABILITY — IG + attention + occlusion + 2D/3D brain viz')
print('=' * 78)

some_meta = next(iter(best_model_meta.values()))
ROI = int(some_meta['roi'])
T_GLOBAL_MAX = 0
ig_roi_sum    = np.zeros(ROI,        dtype=np.float64)
ig_edge_sum   = np.zeros((ROI, ROI), dtype=np.float64)
mean_adj_sum  = np.zeros((ROI, ROI), dtype=np.float64)
roi_occ_sum   = np.zeros(ROI,        dtype=np.float64)
n_subj_total  = 0
ig_window_acc, attn_acc, win_occ_acc = [], [], []

subj_ig_one = subj_attn_one = subj_adj_one = subj_meta_one = None
subj_preds_all, subj_truths_all = [], []

XAI_OUT_DIR = os.path.join(BASE_DIR, 'xai')
os.makedirs(XAI_OUT_DIR, exist_ok=True)

for outer in range(1, N_OUTER + 1):
    inner_pkls_o = _all_inner_pkls(outer)
    for inner in range(1, 5 + 1):
        model = best_models_per_fold[(outer, inner)]
        meta  = best_model_meta[(outer, inner)]
        if model is None: continue

        pkl = torch.load(inner_pkls_o[inner], map_location='cpu',
                          weights_only=False)
        test_arr = pkl['test_graphs']
        test_ds  = SubjectSequenceDatasetV2(test_arr, ROI)
        loader   = make_loader(test_ds, BATCH_SIZE, shuffle=False)
        if len(test_ds) == 0: continue

        all_x, all_adj, all_len, all_y = [], [], [], []
        for x, a, l, y in loader:
            all_x.append(x); all_adj.append(a); all_len.append(l); all_y.append(y)
        x_seq   = torch.cat(all_x, 0).to(_device)
        adj_seq = torch.cat(all_adj, 0).to(_device)
        lengths = torch.cat(all_len, 0).to(_device)
        ys      = torch.cat(all_y, 0).numpy()
        T = adj_seq.shape[1]; T_GLOBAL_MAX = max(T_GLOBAL_MAX, T)

        mean_adj_sum += adj_seq.mean(dim=(0, 1)).cpu().numpy() * len(test_ds)

        with torch.no_grad():
            preds = model(adj_seq, adj_seq, lengths).cpu().numpy()
        preds_unz = preds * meta['y_std'] + meta['y_mean']
        subj_preds_all.extend(preds_unz.tolist())
        subj_truths_all.extend(ys.tolist())

        ig = integrated_gradients(model, adj_seq, lengths, steps=20)
        ig_abs = ig.abs()
        ig_roi_sum  += ig_abs.sum(dim=(1, 3)).sum(dim=0).cpu().numpy()
        ig_edge_sum += ig_abs.sum(dim=1).sum(dim=0).cpu().numpy()
        ig_window_acc.append(ig_abs.sum(dim=(2, 3)).mean(dim=0).cpu().numpy())

        with torch.no_grad():
            _, attn = model.forward_with_attn(adj_seq, adj_seq, lengths)
        attn_acc.append(attn.cpu().numpy().mean(axis=0))

        win_occ_acc.append(window_occlusion(model, adj_seq, lengths).mean(axis=0))
        roi_occ_sum += roi_occlusion(model, adj_seq, lengths).sum(axis=0)
        n_subj_total += len(test_ds)

        if subj_ig_one is None and outer == 1 and inner == 1:
            subj_ig_one   = ig_abs[0].cpu().numpy()
            subj_attn_one = attn[0].cpu().numpy()
            subj_adj_one  = adj_seq[0].cpu().numpy()
            subj_meta_one = {'outer': outer, 'inner': inner, 'idx': 0,
                              'y': float(ys[0]), 'pred': float(preds_unz[0])}

        print('  outer %d inner %d (n=%d)  IG-ROI top: %s' %
              (outer, inner, len(test_ds),
               np.argsort(ig_abs.sum(dim=(1, 3)).sum(dim=0).cpu().numpy())[-5:][::-1].tolist()))

def stack_pad(vecs, T):
    out = np.full((len(vecs), T), np.nan)
    for i, v in enumerate(vecs): out[i, :len(v)] = v
    return out

ig_window_mat = stack_pad(ig_window_acc, T_GLOBAL_MAX)
attn_mat      = stack_pad(attn_acc,       T_GLOBAL_MAX)
win_occ_mat   = stack_pad(win_occ_acc,    T_GLOBAL_MAX)
ig_roi_mean   = ig_roi_sum  / max(1, n_subj_total)
roi_occ_mean  = roi_occ_sum / max(1, n_subj_total)
ig_edge_mean  = ig_edge_sum / max(1, n_subj_total)
mean_adj_mean = mean_adj_sum / max(1, n_subj_total)

ig_top  = np.argsort(ig_roi_mean )[::-1][:20]
occ_top = np.argsort(roi_occ_mean)[::-1][:20]
overlap = sorted(set(ig_top.tolist()) & set(occ_top.tolist()))
print('\nTop-20 ROIs by IG          :', ig_top.tolist())
print('Top-20 ROIs by occlusion   :', occ_top.tolist())
print('Overlap (size %d)          :' % len(overlap), overlap)

# ── Group-level time-series matrix (compute once from a representative fold)
print('  (computing fold-1/inner-1 time-series for plotting)')
pkl1 = torch.load(_all_inner_pkls(1)[1], map_location='cpu', weights_only=False)
ds1  = SubjectSequenceDatasetV2(pkl1['test_graphs'], ROI)
ld1  = make_loader(ds1, BATCH_SIZE, shuffle=False)
all_a = torch.cat([a for _, a, _, _ in ld1], 0)
ts_mat = all_a.abs().mean(dim=3).mean(dim=0).numpy()       # [T, R]

np.save(os.path.join(XAI_OUT_DIR, 'ig_roi_importance.npy'),    ig_roi_mean)
np.save(os.path.join(XAI_OUT_DIR, 'ig_edge_importance.npy'),   ig_edge_mean)
np.save(os.path.join(XAI_OUT_DIR, 'mean_adjacency.npy'),       mean_adj_mean)
np.save(os.path.join(XAI_OUT_DIR, 'roi_occlusion.npy'),        roi_occ_mean)
np.save(os.path.join(XAI_OUT_DIR, 'ig_window_importance.npy'), ig_window_mat)
np.save(os.path.join(XAI_OUT_DIR, 'attn_per_window.npy'),      attn_mat)
np.save(os.path.join(XAI_OUT_DIR, 'window_occlusion.npy'),     win_occ_mat)

coords = get_shen268_coords(n_roi=ROI)

# ── Fig A. summary 2x2 ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
top30 = np.argsort(ig_roi_mean)[::-1][:30]
ax = axes[0, 0]
ax.bar(np.arange(len(top30)) - 0.2,
       ig_roi_mean[top30]   / (ig_roi_mean.max()  + 1e-12),
       width=0.4, label='IG')
ax.bar(np.arange(len(top30)) + 0.2,
       roi_occ_mean[top30]  / (roi_occ_mean.max() + 1e-12),
       width=0.4, label='occlusion')
ax.set_xticks(np.arange(len(top30)))
ax.set_xticklabels(top30, rotation=90, fontsize=7)
ax.set_xlabel('ROI'); ax.set_ylabel('normalised importance')
ax.set_title('Top-30 ROIs: IG vs occlusion'); ax.legend()

ig_win_mean   = np.nanmean(ig_window_mat, axis=0)
attn_mean     = np.nanmean(attn_mat,       axis=0)
win_occ_mean  = np.nanmean(win_occ_mat,    axis=0)
def _norm(v):
    v = np.array(v, dtype=float); s = np.nansum(np.abs(v))
    return v / s if s > 0 else v
xs = np.arange(T_GLOBAL_MAX)
ax = axes[0, 1]
ax.plot(xs, _norm(attn_mean),    label='temporal attention')
ax.plot(xs, _norm(ig_win_mean),  label='IG over windows')
ax.plot(xs, _norm(win_occ_mean), label='window occlusion')
ax.set_xlabel('window'); ax.set_ylabel('normalised importance')
ax.set_title('Temporal importance (avg of 25 folds)'); ax.legend()

ax = axes[1, 0]
im = ax.imshow(ig_edge_mean[np.ix_(ig_top, ig_top)], cmap='magma', aspect='auto')
ax.set_xticks(range(len(ig_top))); ax.set_xticklabels(ig_top, rotation=90, fontsize=7)
ax.set_yticks(range(len(ig_top))); ax.set_yticklabels(ig_top, fontsize=7)
ax.set_title('IG edge importance — top-20 x top-20')
plt.colorbar(im, ax=ax, fraction=0.04)

ax = axes[1, 1]
im = ax.imshow(attn_mat, aspect='auto', cmap='viridis')
ax.set_xlabel('window'); ax.set_ylabel('fold (1..25)')
ax.set_title('Per-fold attention'); plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.savefig(os.path.join(XAI_OUT_DIR, 'xai_summary.png'), dpi=140); plt.close(fig)

# ── Fig B. 2-D brain ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 7))
plot_brain_2d(fig, coords, ig_roi_mean, ig_edge_mean,
               top_k_edges=120, top_label=15)
fig.suptitle('2-D brain (Shen-268) — node colour/size = IG ROI importance, '
              'edge thickness = top IG edges', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(XAI_OUT_DIR, 'brain_2d.png'), dpi=140); plt.close(fig)

# ── Fig C. 3-D brain ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(11, 9))
ax3 = fig.add_subplot(111, projection='3d')
plot_brain_3d(ax3, coords, ig_roi_mean, ig_edge_mean,
               top_k_edges=80, top_label=15)
plt.tight_layout()
plt.savefig(os.path.join(XAI_OUT_DIR, 'brain_3d.png'), dpi=140); plt.close(fig)

# ── Fig D. Correlation matrices ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
vmax = float(np.abs(mean_adj_mean).max()) + 1e-12
im = axes[0].imshow(mean_adj_mean, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                     aspect='auto')
axes[0].set_title('Mean functional connectivity (group-avg adjacency)')
axes[0].set_xlabel('ROI'); axes[0].set_ylabel('ROI')
plt.colorbar(im, ax=axes[0], fraction=0.045)

im = axes[1].imshow(ig_edge_mean, cmap='magma', aspect='auto')
axes[1].set_title('IG edge importance (full matrix)')
axes[1].set_xlabel('ROI'); axes[1].set_ylabel('ROI')
plt.colorbar(im, ax=axes[1], fraction=0.045)
plt.tight_layout()
plt.savefig(os.path.join(XAI_OUT_DIR, 'correlation_matrices.png'), dpi=140)
plt.close(fig)

# ── Fig E. Time-series: overall mean + top-5 ROIs + attention ───────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7),
                          gridspec_kw={'height_ratios': [2, 1]})
ax = axes[0]
overall = ts_mat.mean(axis=1)
ax.plot(np.arange(ts_mat.shape[0]), overall, 'k', lw=2.0,
         label='ROI-mean (all %d ROIs)' % ROI)
top5 = np.argsort(ig_roi_mean)[::-1][:5]
for r in top5:
    ax.plot(np.arange(ts_mat.shape[0]), ts_mat[:, r], lw=1.2, label='ROI %d' % r)
ax.set_xlabel('window'); ax.set_ylabel('mean |connectivity|')
ax.set_title('Per-window connectivity time-series — overall mean + top-5 IG ROIs')
ax.legend(loc='upper right', fontsize=9)

ax = axes[1]
ax.plot(xs, attn_mean / (attn_mean.max() + 1e-12), 'C0', lw=1.6)
ax.fill_between(xs, attn_mean / (attn_mean.max() + 1e-12), alpha=0.3)
ax.set_xlabel('window'); ax.set_ylabel('normalised attention')
ax.set_title('Average temporal attention across 25 folds')
plt.tight_layout()
plt.savefig(os.path.join(XAI_OUT_DIR, 'time_series.png'), dpi=140); plt.close(fig)

# ── Fig F. Subject-level XAI ────────────────────────────────────────────
if subj_ig_one is not None:
    fig, axes = plt.subplots(3, 1, figsize=(16, 11),
                              gridspec_kw={'height_ratios': [3, 1, 1.3]})
    top_roi = np.argsort(subj_ig_one.sum(axis=(0, 2)))[::-1][:30]
    heat = subj_ig_one[:, top_roi, :].sum(axis=2).T
    im = axes[0].imshow(heat, aspect='auto', cmap='magma')
    axes[0].set_yticks(range(len(top_roi)))
    axes[0].set_yticklabels(top_roi, fontsize=8)
    axes[0].set_xlabel('window'); axes[0].set_ylabel('ROI (top 30 by IG)')
    axes[0].set_title('Subject %d (outer %d / inner %d)  true=%.2f  pred=%.2f' %
                       (subj_meta_one['idx'], subj_meta_one['outer'],
                        subj_meta_one['inner'], subj_meta_one['y'],
                        subj_meta_one['pred']))
    plt.colorbar(im, ax=axes[0], fraction=0.025)
    axes[1].plot(np.arange(len(subj_attn_one)), subj_attn_one, lw=1.4)
    axes[1].fill_between(np.arange(len(subj_attn_one)), subj_attn_one, alpha=0.25)
    axes[1].set_ylabel('attention')
    axes[1].set_title('Subject temporal attention')
    for r in top_roi[:5]:
        axes[2].plot(np.arange(subj_adj_one.shape[0]),
                     subj_adj_one[:, r, :].mean(axis=1), label='ROI %d' % r)
    axes[2].set_xlabel('window'); axes[2].set_ylabel('mean |connectivity|')
    axes[2].set_title('Top-5 ROI mean connectivity (subject time series)')
    axes[2].legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(XAI_OUT_DIR, 'subject_xai.png'), dpi=140)
    plt.close(fig)

# ── Fig G. Pred vs truth ────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
arr_p = np.array(subj_preds_all); arr_y = np.array(subj_truths_all)
ax.scatter(arr_y, arr_p, s=18, alpha=0.6)
mn, mx = float(min(arr_y.min(), arr_p.min())), float(max(arr_y.max(), arr_p.max()))
ax.plot([mn, mx], [mn, mx], 'k--', lw=0.8)
r_all = float(np.corrcoef(arr_p, arr_y)[0, 1]) if len(arr_p) > 1 else float('nan')
ax.set_xlabel('true score'); ax.set_ylabel('predicted score')
ax.set_title('Predicted vs true (n=%d, r=%.3f)' % (len(arr_p), r_all))
plt.tight_layout()
plt.savefig(os.path.join(XAI_OUT_DIR, 'pred_vs_truth.png'), dpi=140)
plt.close(fig)

# ── Top-ROI CSV (now includes coordinates) ──────────────────────────────
top_csv = os.path.join(XAI_OUT_DIR, 'top_rois.csv')
with open(top_csv, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['rank', 'ROI_IG', 'IG_score', 'x', 'y', 'z',
                'ROI_OCC', 'OCC_score'])
    for r in range(min(30, len(ig_top))):
        ri = int(ig_top[r])
        w.writerow([r + 1, ri, float(ig_roi_mean[ri]),
                     float(coords[ri, 0]),
                     float(coords[ri, 1]),
                     float(coords[ri, 2]),
                     int(occ_top[r]), float(roi_occ_mean[occ_top[r]])])


# ── Min-max normalise IG importances to [0, 1] ─────────────────────────
def _minmax01(x):
    x = np.asarray(x, dtype=np.float64)
    mn = float(np.nanmin(x)); mx = float(np.nanmax(x))
    if mx - mn < 1e-12: return np.zeros_like(x)
    return (x - mn) / (mx - mn)

ig_roi_norm  = _minmax01(ig_roi_mean)                    # [R]   in [0,1]
roi_occ_norm = _minmax01(roi_occ_mean)                   # [R]   in [0,1]
# Edge normalisation uses only the upper-triangular values (i<j) so symmetry
# is preserved and the matrix's zero diagonal doesn't compress the range.
_iu, _ju  = np.triu_indices(ROI, k=1)
_ev       = ig_edge_mean[_iu, _ju]
_ev_norm  = _minmax01(_ev)
ig_edge_norm = np.zeros_like(ig_edge_mean)
ig_edge_norm[_iu, _ju] = _ev_norm
ig_edge_norm[_ju, _iu] = _ev_norm

# ── Load ROI names + functional networks ────────────────────────────────
roi_names, roi_networks = get_shen268_labels(n_roi=ROI)


# ── Per-run output folder on Google Drive (reuse the runner's folder) ──
if 'RUN_DIR' in globals():
    print('  [drive] reusing run folder from Step 7b:', RUN_DIR)
else:
    import datetime
    _RUN_TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    _DRIVE_BASE = '/content/drive/MyDrive/GNN-mri/runs'
    try:
        os.makedirs(_DRIVE_BASE, exist_ok=True)
        RUN_DIR = os.path.join(_DRIVE_BASE, 'run_17_2_' + _RUN_TS)
        os.makedirs(RUN_DIR, exist_ok=True)
        print('  [drive] saving outputs to', RUN_DIR)
    except Exception:
        RUN_DIR = os.path.join(BASE_DIR, 'run_17_2_' + _RUN_TS)
        os.makedirs(RUN_DIR, exist_ok=True)
        print('  [drive] Drive not mounted — saving locally to', RUN_DIR)

# ── Single consolidated JSON ────────────────────────────────────────────
iu, ju = np.triu_indices(ROI, k=1)
nodes_arr = []
for i in range(ROI):
    nodes_arr.append({
        'roi': i, 'roi_name': roi_names[i], 'network': roi_networks[i],
        'x': float(coords[i, 0]), 'y': float(coords[i, 1]), 'z': float(coords[i, 2]),
        'node_strength':        float(ig_roi_norm[i]),
        'occlusion_importance': float(roi_occ_norm[i]),
    })
edges_arr = []
for k in range(iu.size):
    i, j = int(iu[k]), int(ju[k])
    edges_arr.append({
        'start_node': i, 'end_node': j,
        'start_node_name': roi_names[i], 'end_node_name': roi_names[j],
        'start_network':   roi_networks[i], 'end_network':   roi_networks[j],
        'start_node_strength': float(ig_roi_norm[i]),
        'end_node_strength':   float(ig_roi_norm[j]),
        'edge_importance': float(ig_edge_norm[i, j]),
        'edge_type': ('intra-network'
                       if roi_networks[i] == roi_networks[j]
                       else 'inter-network'),
    })
top_arr = []
for r in range(min(30, len(ig_top))):
    ri = int(ig_top[r])
    top_arr.append({
        'rank': r + 1, 'roi': ri,
        'roi_name': roi_names[ri], 'network': roi_networks[ri],
        'ig_score': float(ig_roi_norm[ri]),
        'x': float(coords[ri, 0]), 'y': float(coords[ri, 1]), 'z': float(coords[ri, 2]),
        'occlusion_top_roi': int(occ_top[r]),
        'occlusion_score':   float(roi_occ_norm[occ_top[r]]),
    })
overall_ts = ts_mat.mean(axis=1)
attn_arr = []
for k in range(attn_mat.shape[0]):
    attn_arr.append({
        'fold': int(k),
        'weights': [None if np.isnan(v) else float(v) for v in attn_mat[k]],
    })

xai_payload = {
    'metadata': {
        'timestamp': _RUN_TS,
        'n_roi': int(ROI),
        'T_windows': int(ts_mat.shape[0]),
        'n_folds': int(attn_mat.shape[0]),
        'primary_metric': str(Config.PRIMARY_METRIC),
        'note': ('All node/edge importance values are min-max '
                 'normalised to [0, 1] across the run.'),
    },
    'nodes':       nodes_arr,
    'edges':       edges_arr,
    'top_rois':    top_arr,
    'timeseries': {
        'T': int(ts_mat.shape[0]),
        'ROI': int(ROI),
        'overall_mean': [float(v) for v in overall_ts],
        'per_roi':      [[float(v) for v in ts_mat[t]]
                          for t in range(ts_mat.shape[0])],
    },
    'attention_per_fold': attn_arr,
}

xai_json_path = os.path.join(RUN_DIR, 'xai_results.json')
with open(xai_json_path, 'w') as f:
    json.dump(xai_payload, f)
print('Wrote consolidated JSON:', xai_json_path)

# Copy PNGs into RUN_DIR too
import shutil
for fn in ['xai_summary.png', 'brain_2d.png', 'brain_3d.png',
           'correlation_matrices.png', 'time_series.png',
           'subject_xai.png', 'pred_vs_truth.png']:
    src = os.path.join(XAI_OUT_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(RUN_DIR, fn))

print('All outputs saved to:', RUN_DIR)
print('  - xai_results.json   (single consolidated XAI export)')
print('  - 7 PNG figures')


## Step 9 - build viewer json

In [ ]:
# ─── Step 10 — Build viewer-schema JSON (matches the web-app spec) ──────────
# Reads the rich xai_results.json that Step 8 wrote and produces a second JSON
# in a flatter, viewer-friendly schema.
#
# Saved alongside xai_results.json as xai_viewer.json
# (xai_results.json stays untouched so the 2-D / 3-D brain plot cells keep
# working — they need coords + network which the viewer schema strips out.)
#
# Output schema:
#   schema_version, model_id, visualisation_id, atlas_id,
#   node_count, edge_count, metrics, edge_selection, plot_edge_selection,
#   nodes[268]      → {id, node_no, score, score_norm}
#   edges[100]      → top 100 by edge_grad DESCENDING (no node filter):
#                       {rank, source, target, source_node_no, target_node_no,
#                        weight, edge_type}
#   plot_edges[30]  → top 30 by edge_grad AMONG the top 15 nodes
#                       (focus-then-filter — matches what 9a / 9b plot)

import json
import numpy as np
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
TOP_EDGES_K = 100   # how many edges in the main "edges" list (overall)
TOP_NODES_K = 15    # focus set for plot_edges — must match TOP_K_STATIC_NODES (9a) and TOP_K_3D_NODES (9b)
TOP_PLOT_K  = 30    # how many edges in plot_edges — must match TOP_K_STATIC_EDGES (9a) and TOP_K_3D_EDGES (9b)

MODEL_ID         = 'tregnnv2'
VISUALISATION_ID = 'tregnnv2_node_edge_importance'
ATLAS_ID         = 'shen268'


# ── Locate this run's xai_results.json ─────────────────────────────────────
if 'RUN_DIR' in globals():
    run_dir = Path(RUN_DIR)
else:
    base = Path('/content/drive/MyDrive/GNN-mri/runs')
    cands = sorted(base.glob('run_*'), key=lambda p: p.stat().st_mtime) if base.exists() else []
    run_dir = cands[-1] if cands else Path('./results/xai')

xai_in_path  = run_dir / 'xai_results.json'
xai_out_path = run_dir / 'xai_viewer.json'
if not xai_in_path.exists():
    raise FileNotFoundError(f'xai_results.json not found in {run_dir}')

print('Reading rich XAI :', xai_in_path)
xai = json.loads(xai_in_path.read_text())
N   = int(xai['metadata']['n_roi'])


# ── Build node scores (consensus_z = signed z-score, score_norm = 0-1) ─────
# v17 saves node_strength (already 0-1 min-max). The viewer schema wants a
# SIGNED "score" (consensus_z) plus a "score_norm" (0-1). We z-score the
# saved node_strength across ROIs to get a signed score centred on 0.
nodes_in = sorted(xai['nodes'], key=lambda n: int(n['roi']))
raw_strength = np.array([float(n['node_strength']) for n in nodes_in],
                          dtype=np.float64)
z          = (raw_strength - raw_strength.mean()) / (raw_strength.std() + 1e-12)
z_min, z_max = float(z.min()), float(z.max())
z_norm01   = (z - z_min) / (z_max - z_min + 1e-12)

nodes_out = [
    {
        'id'        : int(i),
        'node_no'   : int(i + 1),
        'score'     : float(z[i]),
        'score_norm': float(z_norm01[i]),
    }
    for i in range(N)
]


# ── Edge type helper ──────────────────────────────────────────────────────
def _et(t):
    """Collapse 'intra-network'/'within-category' → 'within', else 'between'."""
    return 'within' if t in ('intra-network', 'within-category') else 'between'


# ── edges: top TOP_EDGES_K by edge_importance DESCENDING (no node filter) ──
edges_sorted_all  = sorted(xai['edges'],
                            key=lambda e: -float(e['edge_importance']))
top_edges_overall = edges_sorted_all[:TOP_EDGES_K]

edges_out = []
for r, e in enumerate(top_edges_overall, start=1):
    edges_out.append({
        'rank'           : int(r),
        'source'         : int(e['start_node']),
        'target'         : int(e['end_node']),
        'source_node_no' : int(e['start_node']) + 1,
        'target_node_no' : int(e['end_node'])   + 1,
        'weight'         : float(e['edge_importance']),
        'edge_type'      : _et(e.get('edge_type', '')),
    })


# ── plot_edges: focus-then-filter (same logic as 2D / 3D plot cells) ──────
# Step 1: pick the top TOP_NODES_K nodes by signed score
top_node_ids = set(int(i) for i in np.argsort(z)[::-1][:TOP_NODES_K])

# Step 2: keep edges where BOTH endpoints are in top_node_ids
plot_filtered = [e for e in xai['edges']
                  if int(e['start_node']) in top_node_ids
                  and int(e['end_node'])   in top_node_ids]

# Step 3: rank by edge_importance descending, keep TOP_PLOT_K
plot_sorted = sorted(plot_filtered,
                      key=lambda e: -float(e['edge_importance']))[:TOP_PLOT_K]

plot_edges_out = []
for r, e in enumerate(plot_sorted, start=1):
    plot_edges_out.append({
        'rank'           : int(r),
        'source'         : int(e['start_node']),
        'target'         : int(e['end_node']),
        'source_node_no' : int(e['start_node']) + 1,
        'target_node_no' : int(e['end_node'])   + 1,
        'weight'         : float(e['edge_importance']),
        'edge_type'      : _et(e.get('edge_type', '')),
    })


# ── Assemble viewer JSON ──────────────────────────────────────────────────
viewer = {
    'schema_version'      : 1,
    'model_id'            : MODEL_ID,
    'visualisation_id'    : VISUALISATION_ID,
    'atlas_id'            : ATLAS_ID,
    'node_count'          : N,
    'edge_count'          : len(edges_out),
    'metrics': {
        'node_score'      : 'consensus_z',
        'node_score_norm' : 'consensus_z_norm01',
        'edge_weight'     : 'edge_grad',
    },
    'edge_selection'      : f'top {TOP_EDGES_K} by edge_grad descending',
    'plot_edge_selection' : f'top {TOP_PLOT_K} by edge_grad from among_top_nodes',
    'nodes'      : nodes_out,
    'edges'      : edges_out,
    'plot_edges' : plot_edges_out,
}


# ── Save ──────────────────────────────────────────────────────────────────
with open(xai_out_path, 'w') as f:
    json.dump(viewer, f, indent=2)

print('\nWrote viewer-schema JSON:', xai_out_path)
print(f'  Nodes      : {N}')
print(f'  Edges      : {len(edges_out)}     (top-{TOP_EDGES_K} by edge_grad, overall)')
print(f'  Plot edges : {len(plot_edges_out)}  (top-{TOP_PLOT_K} among top-{TOP_NODES_K} nodes)')
print(f'  Score range: [{float(z.min()):.4f}, {float(z.max()):.4f}]')
print(f'  Score_norm : [0.0, 1.0]   (id={int(np.argmin(z))} ↔ id={int(np.argmax(z))})')

### Step 9a — 4-panel 2-D brain (nilearn glass-brain)

Reads `xai_results.json` from the run folder and renders the **exact same format as the FBNetGen reference image**: `display_mode='lyrz'` (left-sagittal, coronal, right-sagittal, axial), top-15 nodes coloured by IG importance, top-30 edges drawn as blue (within-network) and orange (across-network) lines. Saved to `<RUN_DIR>/brain_2d_4panel.png`.

In [ ]:
# ─── Step 9a — 4-panel 2-D brain (nilearn glass-brain, same format as the
#                FBNetGen reference figure) — reads v17's xai_results.json
#
# Output:  <RUN_DIR>/brain_2d_4panel.png
#
# This is your exact code from the FBNetGen notebook, adapted minimally:
#   * Input source = the consolidated xai_results.json produced by Step 8,
#     not separate node_importance / edge_importance CSVs.
#   * The notebook's variable names are mapped (consensus_z → node_strength,
#     edge_grad → edge_importance, etc.) by building the same DataFrames
#     the FBNetGen code expects.
#   * One figure with 4 anatomical panels (`display_mode='lyrz'`) showing
#     both within-network (blue) and across-network (orange) edges together.
!pip install -q nilearn
!pip install -q nilearn

import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from nilearn import plotting


# ── Hyper-parameters (match the FBNetGen figure) ────────────────────────────
TOP_K_STATIC_NODES   = 15
TOP_K_STATIC_EDGES   = 30
EDGE_SCOPE           = 'among_top_nodes'   # 'among_top_nodes' or 'top_edges_overall'
NODE_INTENSITY_COLORS = ['#eeeeee', '#fff2a8', '#ffd84d', '#f2a900']
NODE_CMAP            = LinearSegmentedColormap.from_list(
                          'fbnetgen_node_gold', NODE_INTENSITY_COLORS)
WITHIN_EDGE_COLOR    = '#2a78c4'    # blue   — within-network
BETWEEN_EDGE_COLOR   = '#d4552d'    # orange — across-network


# ── Locate v17 outputs ──────────────────────────────────────────────────────
if 'RUN_DIR' in globals():
    run_dir = Path(RUN_DIR)
else:
    # Best-effort fallback: latest run_* folder under Drive
    candidates = sorted(
        Path('/content/drive/MyDrive/GNN-mri/runs').glob('run_*'),
        key=lambda p: p.stat().st_mtime) if Path('/content/drive/MyDrive/GNN-mri/runs').exists() else []
    run_dir = candidates[-1] if candidates else Path('./results/xai')

xai_path = run_dir / 'xai_results.json'
if not xai_path.exists():
    raise FileNotFoundError(f'xai_results.json not found in {run_dir}')

print('Loading XAI from:', xai_path)
xai = json.loads(xai_path.read_text())


# ── Build node_df and edge_df with FBNetGen column names ────────────────────
nodes_sorted = sorted(xai['nodes'], key=lambda n: n['roi'])
node_df = pd.DataFrame([{
    'roi_index'         : n['roi'],
    'node_no'           : n['roi'] + 1,
    'consensus_z'       : float(n['node_strength']),         # already 0-1
    'consensus_z_norm01': float(n['node_strength']),
    'network'           : n.get('network', 'Unknown'),
    'roi_label'         : n.get('roi_name', f"ROI_{n['roi']}"),
    'x_mni'             : float(n['x']),
    'y_mni'             : float(n['y']),
    'z_mni'             : float(n['z']),
} for n in nodes_sorted])

edge_df = pd.DataFrame([{
    'roi_src'  : int(e['start_node']),
    'roi_dst'  : int(e['end_node']),
    'edge_grad': float(e['edge_importance']),
    'edge_type': e.get('edge_type', 'inter-network'),       # v17 already classifies
} for e in xai['edges']])

centroids = node_df[['x_mni', 'y_mni', 'z_mni']].to_numpy(dtype=float)


# ── Helper: range-normalise to a numeric band ───────────────────────────────
def _norm(values, low, high):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0: return arr
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or abs(mx - mn) < 1e-12:
        return np.full_like(arr, (low + high) / 2, dtype=float)
    return low + (arr - mn) * (high - low) / (mx - mn)


# ── Pick top nodes and top edges ────────────────────────────────────────────
node_df['_node_score']    = node_df['consensus_z']
node_df['_node_size']     = _norm(node_df['_node_score'].abs().fillna(0), 30, 220)

# Step 1 — focus: pick top 15 nodes by score
top_nodes  = node_df.sort_values('_node_score', ascending=False).head(TOP_K_STATIC_NODES).copy()
shown_rois = set(top_nodes['roi_index'].astype(int))

# Local re-scale across the displayed top nodes (same visual contrast as 3D)
mn, mx = float(top_nodes['_node_score'].min()), float(top_nodes['_node_score'].max())
top_nodes['_node_color01_local'] = (
    0.5 if abs(mx - mn) < 1e-12
    else (top_nodes['_node_score'] - mn) / (mx - mn))
top_nodes['_marker_color'] = [to_hex(NODE_CMAP(float(v)))
                                for v in top_nodes['_node_color01_local']]

# Step 2 — filter: keep only edges where BOTH endpoints are in top 15
edge_plot = edge_df.copy()
if EDGE_SCOPE == 'among_top_nodes':
    edge_plot = edge_plot[edge_plot['roi_src'].isin(shown_rois)
                           & edge_plot['roi_dst'].isin(shown_rois)].copy()
# Step 3 — take top 30 of those by edge importance
edge_plot = edge_plot.sort_values('edge_grad', ascending=False).head(TOP_K_STATIC_EDGES).copy()
edge_plot['_edge_width'] = _norm(edge_plot['edge_grad'].abs(), 1.2, 7.5)


# ── Encode within/between as sign so a diverging cmap can show both ─────────
# within  → POSITIVE values, painted with WITHIN_EDGE_COLOR
# between → NEGATIVE values, painted with BETWEEN_EDGE_COLOR
adj = np.zeros((len(centroids), len(centroids)), dtype=float)
for _, row in edge_plot.iterrows():
    s, d = int(row['roi_src']), int(row['roi_dst'])
    sign = +1.0 if row['edge_type'] in ('intra-network', 'within-category') else -1.0
    w = sign * float(row['edge_grad'])
    adj[s, d] = w; adj[d, s] = w

vmax = float(np.max(np.abs(adj))) + 1e-12
two_color_cmap = LinearSegmentedColormap.from_list(
    'within_between', [BETWEEN_EDGE_COLOR, '#ffffff00', WITHIN_EDGE_COLOR])


# ── Plot the 4-panel glass brain ────────────────────────────────────────────
fig = plt.figure(figsize=(20, 6.5), facecolor='white')
display = plotting.plot_connectome(
    adjacency_matrix=adj,
    node_coords=centroids,
    node_size=0,                             # we'll add markers manually
    edge_threshold=None,
    edge_cmap=two_color_cmap,
    edge_vmin=-vmax, edge_vmax=+vmax,
    edge_kwargs={'linewidth': 1.5, 'alpha': 0.78},
    display_mode='lyrz',                      # 4-panel: L-sag, coronal, R-sag, axial
    colorbar=False,
    title=f'XAI brain map — top {len(top_nodes)} nodes, top {len(edge_plot)} edges',
    figure=fig,
    axes=(0.02, 0.05, 0.85, 0.92),            # leave room on the right for node bar
)

# Top-node markers on top of the glass brain
display_coords = centroids[top_nodes['roi_index'].to_numpy()]
display.add_markers(
    marker_coords=display_coords,
    marker_color=top_nodes['_marker_color'].tolist(),
    marker_size=top_nodes['_node_size'].to_numpy(),
)

# Node-importance colorbar
sm = plt.cm.ScalarMappable(cmap=NODE_CMAP, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cax = fig.add_axes([0.905, 0.22, 0.013, 0.56])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Node importance', labelpad=10)
cbar.set_ticks([0.0, 0.5, 1.0])
cbar.set_ticklabels(['Lower', 'Mid', 'Higher'])
cbar.ax.tick_params(labelsize=9, pad=2)

# Edge-color legend
legend_handles = [
    Line2D([0],[0], color=BETWEEN_EDGE_COLOR, lw=2.5, label='Across categories'),
    Line2D([0],[0], color=WITHIN_EDGE_COLOR,  lw=2.5, label='Within category'),
]
fig.legend(handles=legend_handles, loc='lower right',
            bbox_to_anchor=(0.99, 0.05), frameon=False, fontsize=10)

out_path = run_dir / 'brain_2d_4panel.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
plt.close('all')
print('Saved 4-panel 2-D brain :', out_path)


### Step 9b — Interactive 3-D brain (plotly HTML)

Uses the *exact* FBNetGen plotly + nilearn-surface code, fed from the v17 `xai_results.json`. Output is an **interactive HTML** (`<RUN_DIR>/brain_3d.html`) — open it in a browser to rotate, hover, and filter by category. Brain mesh is `fsaverage5` pial surface at α = 0.16; node sizes from |consensus_z|; node colour locally rescaled across the top-15.

In [ ]:
# ─── Step 9b — Interactive 3-D brain HTML (plotly + nilearn surface) ───────
#
# Output:  <RUN_DIR>/brain_3d.html
#
# This is your exact code from the FBNetGen notebook, adapted minimally:
#   * Input source = the consolidated xai_results.json produced by Step 8,
#     not separate node_importance / edge_importance CSVs.
#   * The notebook's variable names are mapped (consensus_z → node_strength,
#     edge_grad → edge_importance, etc.) by building the same DataFrames
#     the FBNetGen code expects, then the original code runs unchanged.
#   * Saves an interactive HTML file (NOT a PNG) using plotly.

from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go


# ── Hyper-parameters (identical to your FBNetGen 3D figure) ────────────────
TOP_K_3D_NODES            = 15
TOP_K_3D_EDGES            = 30
NODE_IMPORTANCE_METRIC    = 'consensus_z'         # mapped from node_strength
EDGE_IMPORTANCE_METRIC    = 'edge_grad'           # mapped from edge_importance
NODE_CATEGORY_COL         = 'network'
EDGE_SCOPE                = 'among_top_nodes'
NODE_INTENSITY_COLORSCALE = [
    [0.0,  '#eeeeee'],
    [0.35, '#fff2a8'],
    [0.7,  '#ffd84d'],
    [1.0,  '#f2a900'],
]
SHOW_BRAIN_SURFACE     = True
BRAIN_SURFACE_OPACITY  = 0.16
SHOW_ALL_ROIS_FAINT    = False


# ── Locate this run's xai_results.json ────────────────────────────────────
if 'RUN_DIR' in globals():
    out_dir = Path(RUN_DIR)
else:
    base = Path('/content/drive/MyDrive/GNN-mri/runs')
    cands = sorted(base.glob('run_*'), key=lambda p: p.stat().st_mtime) if base.exists() else []
    out_dir = cands[-1] if cands else Path('./results/xai')
xai_path = out_dir / 'xai_results.json'
if not xai_path.exists():
    raise FileNotFoundError(f'xai_results.json not found in {out_dir}')
target_suffix = out_dir.name                                # e.g. run_20260512_103045

print('Loading XAI from:', xai_path)
xai = json.loads(xai_path.read_text())


# ── Build the DataFrames the original FBNetGen code expects ────────────────
nodes_sorted = sorted(xai['nodes'], key=lambda n: n['roi'])
node_df = pd.DataFrame([{
    'roi_index'         : n['roi'],
    'node_no'           : n['roi'] + 1,
    'consensus_z'       : float(n['node_strength']),         # 0-1 in v17
    'consensus_z_norm01': float(n['node_strength']),
    'network'           : n.get('network', 'Unknown'),
    'roi_label'         : n.get('roi_name', f"ROI_{n['roi']}"),
} for n in nodes_sorted])

edge_df = pd.DataFrame([{
    'roi_src'   : int(e['start_node']),
    'roi_dst'   : int(e['end_node']),
    'edge_grad' : float(e['edge_importance']),
    '_edge_type': ('within-category'
                    if e.get('edge_type','') in ('intra-network','within-category')
                    else 'between-category'),
    '_src_cat'  : e.get('start_network', 'Unknown'),
    '_dst_cat'  : e.get('end_network',   'Unknown'),
} for e in xai['edges']])

centroids = np.array([[float(n['x']), float(n['y']), float(n['z'])]
                       for n in nodes_sorted], dtype=float)


# ─────────────────────────────────────────────────────────────────────────────
# === BELOW: your original FBNetGen 3-D plot code, unchanged in structure ===
# ─────────────────────────────────────────────────────────────────────────────
def _norm(values, low, high):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0: return arr
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or abs(mx - mn) < 1e-12:
        return np.full_like(arr, (low + high) / 2, dtype=float)
    return low + (arr - mn) * (high - low) / (mx - mn)

def _clean_category(v):
    if pd.isna(v) or not str(v).strip() or str(v).lower() == 'nan':
        return 'Unknown'
    return str(v).strip()

# -------------------------
# Nodes
# -------------------------
node_df = node_df.copy()
node_df['roi_index']    = node_df['roi_index'].astype(int)
node_df['consensus_z']  = pd.to_numeric(node_df['consensus_z'], errors='coerce').fillna(0)
node_df['consensus_z_norm01'] = pd.to_numeric(node_df['consensus_z_norm01'],
                                              errors='coerce').fillna(0).clip(0, 1)
node_df['_node_score_raw'] = node_df['consensus_z']
node_df['_node_score']     = node_df['consensus_z']
node_df['_node_magnitude'] = node_df['_node_score'].abs()
node_df['_node_size']      = _norm(node_df['_node_magnitude'].fillna(0), 7, 24)
node_df['_category']       = node_df[NODE_CATEGORY_COL].map(_clean_category)

top_nodes  = node_df.sort_values('_node_score', ascending=False).head(TOP_K_3D_NODES).copy()
shown_rois = set(top_nodes['roi_index'].astype(int))
node_lookup = node_df.set_index('roi_index').to_dict('index')
categories = sorted(top_nodes['_category'].dropna().unique().tolist())

color_min = float(top_nodes['_node_score'].min())
color_max = float(top_nodes['_node_score'].max())
if abs(color_max - color_min) < 1e-12:
    top_nodes['_node_color01_local'] = 0.5
else:
    top_nodes['_node_color01_local'] = (
        (top_nodes['_node_score'] - color_min) / (color_max - color_min))
cmin, cmax = 0.0, 1.0

# -------------------------
# Edges
# -------------------------
edge_plot = edge_df.copy()
edge_plot['_edge_raw']   = pd.to_numeric(edge_plot['edge_grad'], errors='coerce').fillna(0)
edge_plot['_edge_score'] = edge_plot['_edge_raw']
if EDGE_SCOPE == 'among_top_nodes':
    edge_plot = edge_plot[edge_plot['roi_src'].isin(shown_rois)
                           & edge_plot['roi_dst'].isin(shown_rois)].copy()
if len(edge_plot) == 0:
    print('No edges among top nodes; falling back to top_edges_overall.')
    edge_plot = edge_df.copy()
    edge_plot['_edge_raw']   = pd.to_numeric(edge_plot['edge_grad'],
                                              errors='coerce').fillna(0)
    edge_plot['_edge_score'] = edge_plot['_edge_raw']
edge_plot = edge_plot.sort_values('_edge_score', ascending=False).head(TOP_K_3D_EDGES).copy()
edge_plot['_edge_width'] = _norm(edge_plot['_edge_score'].abs().fillna(0), 1.2, 9.0)

# -------------------------
# Plot
# -------------------------
traces, trace_filters = [], []

if SHOW_BRAIN_SURFACE:
    try:
        from nilearn import datasets, surface
        fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
        for hemi, surf_path in [('left',  fsaverage.pial_left),
                                  ('right', fsaverage.pial_right)]:
            coords, faces = surface.load_surf_mesh(surf_path)
            traces.append(go.Mesh3d(
                x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color='#d8d2c4', opacity=BRAIN_SURFACE_OPACITY,
                name=f'{hemi} cortical surface',
                hoverinfo='skip', showscale=False,
                lighting=dict(ambient=0.55, diffuse=0.65, specular=0.12, roughness=0.8),
                lightposition=dict(x=0, y=-120, z=120),
                showlegend=True,
            ))
            trace_filters.append({'kind': 'brain', 'cats': set(categories)})
    except Exception as e:
        print(f'Could not render cortical surface ({type(e).__name__}: {e}). '
              'Continuing with nodes/edges only.')

if SHOW_ALL_ROIS_FAINT:
    traces.append(go.Scatter3d(
        x=centroids[:, 0], y=centroids[:, 1], z=centroids[:, 2],
        mode='markers',
        marker=dict(size=3, color='lightgray', opacity=0.16),
        hoverinfo='skip', name='All Shen-268 ROIs', showlegend=True))
    trace_filters.append({'kind': 'background', 'cats': set(categories)})

for cat in categories:
    sub = top_nodes[top_nodes['_category'] == cat].copy()
    idx = sub['roi_index'].astype(int).to_numpy()
    xyz = centroids[idx]
    hover = []
    for _, row in sub.iterrows():
        node_no = (int(row['node_no']) if 'node_no' in row and pd.notna(row['node_no'])
                     else int(row['roi_index']) + 1)
        parts = [
            f'ROI {node_no}',
            f'Category: {row["_category"]}',
            f'roi_index={int(row["roi_index"])}',
            f'consensus_z={float(row["consensus_z"]):.4f}',
            f'consensus_z_norm01={float(row["consensus_z_norm01"]):.4f}',
            f'local_color01={float(row["_node_color01_local"]):.4f}',
        ]
        for c in ['roi_label', 'region_name', 'region_detail', 'hemisphere', 'lobe']:
            if c in sub.columns and pd.notna(row.get(c, '')) and str(row[c]).strip():
                parts.append(f'{c}: {row[c]}')
        hover.append('<br>'.join(parts))
    traces.append(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='markers+text',
        text=[str(int(n)) for n in sub.get('node_no', sub['roi_index'] + 1)],
        textposition='top center',
        hovertext=hover, hoverinfo='text',
        marker=dict(
            size=sub['_node_size'], color=sub['_node_color01_local'],
            cmin=cmin, cmax=cmax,
            colorscale=NODE_INTENSITY_COLORSCALE,
            colorbar=dict(title='local color<br>0-1'),
            showscale=len([t for t in trace_filters if t.get('kind') == 'node']) == 0,
            opacity=0.94, line=dict(width=1, color='black'),
        ),
        name=f'Nodes: {cat}', showlegend=True,
    ))
    trace_filters.append({'kind': 'node', 'cats': {cat}})

edge_type_seen = set()
for _, row in edge_plot.iterrows():
    s, d = int(row['roi_src']), int(row['roi_dst'])
    sx, sy, sz = centroids[s]; dx, dy, dz = centroids[d]
    edge_type = row['_edge_type']
    color = '#2a78c4' if edge_type == 'within-category' else '#d4552d'
    src_no = int(node_lookup.get(s, {}).get('node_no', s + 1))
    dst_no = int(node_lookup.get(d, {}).get('node_no', d + 1))
    hover = '<br>'.join([
        f'ROI {src_no} -> ROI {dst_no}',
        f'Edge type: {edge_type}',
        f'Source category: {row["_src_cat"]}',
        f'Destination category: {row["_dst_cat"]}',
        f'{EDGE_IMPORTANCE_METRIC}: {float(row["_edge_raw"]):.6g}',
    ])
    traces.append(go.Scatter3d(
        x=[sx, dx], y=[sy, dy], z=[sz, dz],
        mode='lines',
        line=dict(color=color, width=float(row['_edge_width'])),
        opacity=0.72,
        hovertext=hover, hoverinfo='text',
        name=edge_type, legendgroup=edge_type,
        showlegend=edge_type not in edge_type_seen,
    ))
    edge_type_seen.add(edge_type)
    trace_filters.append({'kind': 'edge', 'cats': {row['_src_cat'], row['_dst_cat']}})

fig = go.Figure(data=traces)

buttons = [dict(label='All categories', method='update',
                  args=[{'visible': [True] * len(traces)}])]
for cat in categories:
    visible = []
    for f in trace_filters:
        if   f['kind'] == 'brain':      visible.append(SHOW_BRAIN_SURFACE)
        elif f['kind'] == 'background': visible.append(SHOW_ALL_ROIS_FAINT)
        else: visible.append(cat in f['cats'])
    buttons.append(dict(label=cat, method='update', args=[{'visible': visible}]))

fig.update_layout(
    title=f'3D brain surface with Shen-268 importance graph: '
          f'top {TOP_K_3D_NODES} nodes, top {len(edge_plot)} edges',
    scene=dict(
        xaxis=dict(visible=False, showbackground=False, showgrid=False,
                    zeroline=False, showticklabels=False, title=''),
        yaxis=dict(visible=False, showbackground=False, showgrid=False,
                    zeroline=False, showticklabels=False, title=''),
        zaxis=dict(visible=False, showbackground=False, showgrid=False,
                    zeroline=False, showticklabels=False, title=''),
        bgcolor='rgba(0,0,0,0)',
        aspectmode='data',
        camera=dict(eye=dict(x=1.7, y=1.7, z=1.25)),
    ),
    updatemenus=[dict(buttons=buttons, direction='down', x=0.01, y=0.98,
                       xanchor='left', yanchor='top')],
    legend=dict(x=0.01, y=0.82),
    margin=dict(l=0, r=0, t=58, b=0),
    height=820,
)
fig.show()

html_path = out_dir / f'brain_3d_top{TOP_K_3D_NODES}_{TOP_K_3D_EDGES}_{target_suffix}.html'
fig.write_html(str(html_path), include_plotlyjs='cdn')
print('Saved interactive 3D brain HTML:', html_path)
print(f'Plotted {len(top_nodes)} nodes and {len(edge_plot)} edges.')


### Check viewer json and 2D, 3D brain

In [ ]:
# ─── Verification cell — check xai_viewer.json against xai_results.json ────
# Confirms:
#   1. nodes list has 268 entries
#   2. score_norm is in [0, 1]
#   3. edges list has 100 entries, ranked correctly by weight desc (no node filter)
#   4. plot_edges list has 30 entries, all endpoints in top-15 nodes,
#      ranked correctly by weight desc
#   5. The top-15 nodes I'd pick (by signed score) match the focus set used
#      for plot_edges
#   6. Reports overlap with what 2D / 3D would have plotted

import json
import numpy as np
from pathlib import Path

# ── Locate JSONs ──────────────────────────────────────────────────────────
if 'RUN_DIR' in globals():
    run_dir = Path(RUN_DIR)
else:
    base = Path('/content/drive/MyDrive/GNN-mri/runs')
    cands = sorted(base.glob('run_*'), key=lambda p: p.stat().st_mtime) if base.exists() else []
    run_dir = cands[-1] if cands else Path('./results/xai')

xai_path    = run_dir / 'xai_results.json'
viewer_path = run_dir / 'xai_viewer.json'
assert xai_path.exists(),    f'xai_results.json missing in {run_dir}'
assert viewer_path.exists(), f'xai_viewer.json missing in {run_dir}'

xai    = json.loads(xai_path.read_text())
viewer = json.loads(viewer_path.read_text())

# Local copies of the constants used by Steps 10 / 9a / 9b
TOP_EDGES_K = 100
TOP_NODES_K = 15
TOP_PLOT_K  = 30

results = []
def check(name, cond, detail=''):
    status = 'PASS' if cond else 'FAIL'
    results.append((status, name, detail))
    print(f'  [{status}]  {name}' + (f'  — {detail}' if detail else ''))

print('=' * 72)
print(f'Verifying: {viewer_path}')
print('=' * 72)


# ── Reproduce the node scores the same way Step 10 did ───────────────────
nodes_in     = sorted(xai['nodes'], key=lambda n: int(n['roi']))
raw_strength = np.array([float(n['node_strength']) for n in nodes_in],
                          dtype=np.float64)
z      = (raw_strength - raw_strength.mean()) / (raw_strength.std() + 1e-12)
mn, mx = float(z.min()), float(z.max())
z_norm = (z - mn) / (mx - mn + 1e-12)
expected_top_node_ids = set(int(i) for i in np.argsort(z)[::-1][:TOP_NODES_K])

# Helper — unordered edge as a (min, max) tuple
def _edge_key(e):
    a, b = int(e.get('source', e.get('start_node'))), int(e.get('target', e.get('end_node')))
    return (min(a, b), max(a, b))


# ── 1. node count + score_norm range ──────────────────────────────────────
print('\n[Section 1] nodes')
check('node_count == 268',                len(viewer['nodes']) == 268,
       f"got {len(viewer['nodes'])}")
check('every score_norm in [0, 1]',
       all(0.0 - 1e-9 <= n['score_norm'] <= 1.0 + 1e-9 for n in viewer['nodes']))
check('id == node_no - 1 for every node',
       all(int(n['id']) + 1 == int(n['node_no']) for n in viewer['nodes']))


# ── 2. edges (top 100 overall, NOT filtered by nodes) ────────────────────
print('\n[Section 2] edges (top-100 overall)')
edges_v = viewer['edges']
check(f'edge count == {TOP_EDGES_K}', len(edges_v) == TOP_EDGES_K,
       f'got {len(edges_v)}')

# Reproduce: top TOP_EDGES_K by edge_importance descending
expected_top_overall = sorted(xai['edges'],
                               key=lambda e: -float(e['edge_importance']))[:TOP_EDGES_K]
expected_set_overall = {_edge_key(e) for e in expected_top_overall}
actual_set_overall   = {_edge_key(e) for e in edges_v}
check('edges set matches top-100 by edge_importance descending',
       expected_set_overall == actual_set_overall,
       f'overlap {len(expected_set_overall & actual_set_overall)}/'
       f'{len(expected_set_overall)}')

# Weight monotone non-increasing
weights = [float(e['weight']) for e in edges_v]
check('edges weights monotone non-increasing',
       all(weights[i] >= weights[i + 1] for i in range(len(weights) - 1)))

# rank field correct
check('ranks 1..N in order',
       [e['rank'] for e in edges_v] == list(range(1, len(edges_v) + 1)))

# This list should NOT be restricted to top-15 nodes
overall_endpoint_ids = {int(e['source']) for e in edges_v} | {int(e['target']) for e in edges_v}
nodes_outside_focus  = overall_endpoint_ids - expected_top_node_ids
check('edges include endpoints OUTSIDE top-15 (no node filter)',
       len(nodes_outside_focus) > 0,
       f'{len(nodes_outside_focus)} non-top-15 ROIs appear in edges')


# ── 3. plot_edges (top 30 among top 15 nodes) ────────────────────────────
print('\n[Section 3] plot_edges (focus-then-filter)')
plot_v = viewer['plot_edges']
check(f'plot_edge count == {TOP_PLOT_K}', len(plot_v) == TOP_PLOT_K,
       f'got {len(plot_v)}')

# Reproduce: focus → filter → top K
expected_filtered = [e for e in xai['edges']
                     if int(e['start_node']) in expected_top_node_ids
                     and int(e['end_node'])   in expected_top_node_ids]
expected_plot_top = sorted(expected_filtered,
                            key=lambda e: -float(e['edge_importance']))[:TOP_PLOT_K]
expected_plot_set = {_edge_key(e) for e in expected_plot_top}
actual_plot_set   = {_edge_key(e) for e in plot_v}
check('plot_edges set matches focus-then-filter recomputation',
       expected_plot_set == actual_plot_set,
       f'overlap {len(expected_plot_set & actual_plot_set)}/'
       f'{len(expected_plot_set)}')

# Every endpoint must be in top-15
plot_endpoint_ids = {int(e['source']) for e in plot_v} | {int(e['target']) for e in plot_v}
strays = plot_endpoint_ids - expected_top_node_ids
check('every plot_edges endpoint is in top-15',
       len(strays) == 0,
       f'strays: {sorted(strays)}' if strays else '')

# Weight monotone non-increasing
pw = [float(e['weight']) for e in plot_v]
check('plot_edges weights monotone non-increasing',
       all(pw[i] >= pw[i + 1] for i in range(len(pw) - 1)))

# rank field correct
check('plot_edges ranks 1..N in order',
       [e['rank'] for e in plot_v] == list(range(1, len(plot_v) + 1)))


# ── 4. Top-15 focus set inferred from plot_edges ─────────────────────────
print('\n[Section 4] top-15 focus consistency')
print(f'  expected top-15 ROI ids (by score) : {sorted(expected_top_node_ids)}')
print(f'  ROIs appearing in plot_edges       : {sorted(plot_endpoint_ids)}')
missing_from_plot = expected_top_node_ids - plot_endpoint_ids
if missing_from_plot:
    print(f'  Note: {len(missing_from_plot)} top-15 ROIs have no edges in plot_edges '
          f'({sorted(missing_from_plot)}). This is OK — they will still be plotted '
          f'as standalone markers by 9a / 9b.')
else:
    print('  All 15 top nodes have at least one edge in plot_edges.')


# ── 5. Summary ───────────────────────────────────────────────────────────
print('\n' + '=' * 72)
failed = [r for r in results if r[0] == 'FAIL']
if failed:
    print(f'  {len(failed)} CHECK(S) FAILED:')
    for _, name, detail in failed:
        print(f'   - {name}  ({detail})')
else:
    print(f'  All {len(results)} checks passed.')
print('=' * 72)

## Step 10 — Visualize 25-fold Results

Two-panel figure saved to the SAME Drive run folder Step 7b/8 used (`/content/drive/MyDrive/fyp/runs/run_<TIMESTAMP>/summary_25folds.png`):

- **Left** — aggregate scatter of all 250 test-subject predictions vs actual, with overall MAE / RMSE / Pearson / Spearman / R / R².
- **Right** — 25-bar chart of per-fold Pearson r and Spearman ρ (`o1i1` … `o5i5`) with average lines.

In [ ]:
# ─── Step 10 — Visualize Results (all 25 folds) ──────────────────────────────
# Renders a 2-panel figure summarising the v17 25-fold run:
#   Left  : subject-aggregate scatter of all 250 test subjects (pred vs truth)
#   Right : per-fold Pearson r and Spearman ρ bars (25 folds)
# Saved to the same Drive run folder as Step 7b / Step 8.

import os, numpy as np
import matplotlib.pyplot as plt

# Reuse RUN_DIR from Step 7b / Step 8 if present, else create a new local folder
if 'RUN_DIR' not in globals():
    import datetime as _dt
    _RUN_TS = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
    try:
        os.makedirs('/content/drive/MyDrive/GNN-mri/runs', exist_ok=True)
        RUN_DIR = os.path.join('/content/drive/MyDrive/GNN-mri/runs', 'run_17_2_' + _RUN_TS)
    except Exception:
        RUN_DIR = os.path.join(BASE_DIR, 'run_17_2_' + _RUN_TS)
    os.makedirs(RUN_DIR, exist_ok=True)

# ── Build per-fold lists ────────────────────────────────────────────────────
_pm = Config.PRIMARY_METRIC
fold_labels, prs_list, spr_list, mae_list = [], [], [], []
for o in range(1, N_OUTER + 1):
    for i in range(1, 5 + 1):
        m = all_results[o][i]['test_metrics']
        fold_labels.append(f'o{o}i{i}')
        prs_list.append(m['pearson'])
        spr_list.append(m['spearman'])
        mae_list.append(m['mae'])

# Aggregate over all 250 test subjects
agg_p = np.array(agg_subj_preds_25)
agg_y = np.array(agg_subj_ys_25)
agg_m = all_metrics(agg_p, agg_y)

# ── Figure ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

# Left: aggregate scatter
ax = axes[0]
ax.scatter(agg_y, agg_p, alpha=0.55, s=35, c='steelblue', edgecolor='black',
            linewidth=0.3)
lo, hi = float(min(agg_y.min(), agg_p.min())), float(max(agg_y.max(), agg_p.max()))
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect (y = x)')

def _ml(lbl, v, prim):
    return f"{lbl}={v:.3f}{'  ← PRIMARY' if prim else ''}"
txt = (_ml("MAE",     agg_m['mae'],     _pm == 'mae')     + "    " +
       _ml("RMSE",    agg_m['rmse'],    _pm == 'rmse')    + "\n" +
       _ml("Pearson", agg_m['pearson'], _pm == 'pearson') + "    " +
       _ml("Spearman",agg_m['spearman'],_pm == 'spearman')+ "\n" +
       _ml("R",       agg_m['r'],       _pm == 'r')       + "    " +
       _ml("R2",      agg_m['r2'],      _pm == 'r2'))
ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow', alpha=0.85))
ax.set_xlabel('Actual score')
ax.set_ylabel('Predicted score')
ax.set_title(f'Aggregate scatter across all 25 folds  (n = {len(agg_y)} subjects)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Right: per-fold bars
ax2 = axes[1]
x = np.arange(len(prs_list))
ax2.bar(x - 0.2, prs_list, 0.35, color='steelblue', alpha=0.85, label='Pearson r')
ax2.bar(x + 0.2, spr_list, 0.35, color='darkorange', alpha=0.85, label='Spearman ρ')
ax2.axhline(np.nanmean(prs_list), color='steelblue', ls='--', lw=1.2,
            label=f'Avg Pearson = {np.nanmean(prs_list):.3f}')
ax2.axhline(np.nanmean(spr_list), color='darkorange', ls='--', lw=1.2,
            label=f'Avg Spearman = {np.nanmean(spr_list):.3f}')
ax2.axhline(0, color='black', lw=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(fold_labels, rotation=90, fontsize=7)
ax2.set_ylabel('Correlation')
ax2.set_title('Per-fold correlation (25 folds)')
ax2.legend(loc='lower right', fontsize=8)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(-1.05, 1.05)

fig.suptitle(f'T-RegGNN v17 — 25-fold summary  (primary metric: {_pm})',
              fontsize=13)
fig.tight_layout()

fig_path = os.path.join(RUN_DIR, 'summary_25folds.png')
plt.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)


## Save Model

In [ ]:
# ─── Save the 25 trained models that are already in memory ─────────────────
# Run this after Step 7b WITHOUT re-running training. It uses
#   best_models_per_fold, best_model_meta, all_results, Config
# which Step 7b leaves behind in the notebook namespace.

import os, shutil, torch, numpy as np

# ── 1. Resolve or create the run folder + models/ sub-folder ──────────────
if 'RUN_DIR' not in globals():
    import datetime as _dt
    _RUN_TS = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
    _drive  = '/content/drive/MyDrive/GNN-mri/runs'
    try:
        os.makedirs(_drive, exist_ok=True)
        RUN_DIR = os.path.join(_drive, 'run_' + _RUN_TS)
    except Exception:
        RUN_DIR = os.path.join('./results', 'run_' + _RUN_TS)
    os.makedirs(RUN_DIR, exist_ok=True)
MODELS_DIR = os.path.join(RUN_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print('RUN_DIR    :', RUN_DIR)
print('MODELS_DIR :', MODELS_DIR)

# ── 2. Defaults for hyper-params that inference.py records (if missing) ────
_gnn_h    = int(globals().get('GNN_H',    8))
_attn_h   = int(globals().get('ATTN_H',   8))
_dropout  = float(globals().get('DROP',   0.50))
_pmetric  = str(Config.PRIMARY_METRIC) if 'Config' in globals() else 'pearson'

# ── 3. Save every (outer, inner) fold ──────────────────────────────────────
saved_paths = {}
for (outer, inner), model in best_models_per_fold.items():
    meta = best_model_meta.get((outer, inner), {})
    m    = all_results[outer][inner]['test_metrics']
    p    = os.path.join(MODELS_DIR,
                          f'best_model_outer{outer}_inner{inner}.pt')
    torch.save({
        'model_state_dict': model.state_dict(),
        'y_mean':           float(meta.get('y_mean', 0.0)),
        'y_std':            float(meta.get('y_std',  1.0)),
        'roi':              int(meta.get('roi', model.n_roi)),
        'outer':            int(outer),
        'inner':            int(inner),
        'primary_metric':   _pmetric,
        'test_metrics':     m,
        'architecture':     'TRegGNNv2 (DenseGCN + temporal-attention)',
        'gnn_hidden':       _gnn_h,
        'attn_hidden':      _attn_h,
        'dropout':          _dropout,
    }, p)
    saved_paths[(outer, inner)] = p
    print(f'  saved: outer{outer:>2d} inner{inner:>2d}  '
          f'Pearson={m["pearson"]:+.4f}  →  {os.path.basename(p)}')

# ── 4. Identify the champion and copy it to RUN_DIR/best_model.pt ──────────
flat = [(o, i, all_results[o][i]['test_metrics'])
         for (o, i) in saved_paths]
prs  = [m['pearson'] for (_, _, m) in flat]
best_idx = int(np.nanargmax(prs))
best_o, best_i, best_m = flat[best_idx]
src = saved_paths[(best_o, best_i)]
dst = os.path.join(RUN_DIR, 'best_model.pt')
shutil.copy2(src, dst)

print('\n' + '=' * 60)
print(f'Champion fold : outer {best_o}  inner {best_i}')
print(f'Pearson r     : {best_m["pearson"]:+.4f}')
print(f'Spearman ρ    : {best_m["spearman"]:+.4f}')
print(f'MAE           : {best_m["mae"]:.4f}')
print(f'Copied to     : {dst}')
print(f'\nUse with inference.py:')
print(f'  python inference.py --model "{dst}" \\')
print(f'                       --subject <subject.npy> --out result.json')
print('=' * 60)
print(f'Saved {len(saved_paths)} fold checkpoints + 1 champion to {RUN_DIR}')


## Check y

In [ ]:
import torch
# Pick any one of your training PKLs
pkl = torch.load('/content/drive/MyDrive/GNN-mri/folds_data/'
                 'graphs_outer1_inner1.pkl',
                 map_location='cpu', weights_only=False)
train_arr = pkl['train_graphs']

# Print the y of the first 10 subjects
import numpy as np
ys = [float(train_arr[s, 0].y) for s in range(min(10, train_arr.shape[0]))]
print('First 10 subjects y values:', ys)
print('range:', min(ys), 'to', max(ys))
print('mean:', np.mean(ys), 'std:', np.std(ys))

# Print any attribute names the Data object has — sometimes the label is in there
g = train_arr[0, 0]
print('PyG Data attributes:', [k for k in dir(g) if not k.startswith('_')])